In [1]:
import pandas as pd
import numpy as np
import re
from pyspark.sql import SparkSession
import sys



StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 3, Finished, Available, Finished, False)

In [2]:
import sys

# 1. Rice series details — confirm unit, country, coverage
df_te = spark.table("dbo.gld_te_commodity_prices")
df_rice = df_te.filter(df_te.Code == "RR1:COM")

print("=== Rice (RR1:COM) — distinct Name/Unit/Country/Origin ===")
df_rice.select("Name","Unit","Country","Origin").distinct().toPandas().to_csv(sys.stdout, index=False)

print("\n=== Rice date range & row count ===")
df_rice.selectExpr(
    "min(to_date(Date,'MM/dd/yyyy')) as min_date",
    "max(to_date(Date,'MM/dd/yyyy')) as max_date",
    "count(*) as n"
).show()

print("\n=== Rice sample rows ===")
df_rice.orderBy(df_rice.Date.desc()).limit(10).toPandas().to_csv(sys.stdout, index=False)

# 2. All series available per commodity in the international prices table (still needed)
df_intl = spark.table("dbo.gld_international_prices")
print("\n=== Distinct Commodity/Name/Unit/Country/Origin combos (Wheat/Corn/Soybean/Barley) ===")
df_intl.filter(df_intl.Commodity.isin("Wheat","Corn","Soybean","Barley","Soybeans","Soy")) \
       .select("Commodity","Name","Unit","Country","Origin").distinct() \
       .toPandas().to_csv(sys.stdout, index=False)

# 3. Check whether futures-contract columns are populated (mixed spot+futures data)
print("\n=== Futures-contract column population check ===")
df_intl.filter(df_intl.Commodity.isin("Wheat","Corn","Soybean","Barley","Soybeans","Soy")) \
       .groupBy("Commodity") \
       .agg(
           {"shipment_delivery_month":"count", "underlying_futures_contract":"count", "Date":"count"}
       ).show(truncate=False)

# 4. Date range per commodity in international prices
print("\n=== Date range per commodity ===")
for c in ["Wheat","Corn","Soybean","Barley","Soybeans","Soy"]:
    sub = df_intl.filter(df_intl.Commodity == c)
    n = sub.count()
    if n > 0:
        sub.selectExpr(f"'{c}' as commodity", "min(Date) as min_date", "max(Date) as max_date", "count(*) as n").show()

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 4, Finished, Available, Finished, False)

=== Rice (RR1:COM) — distinct Name/Unit/Country/Origin ===
Name,Unit,Country,Origin
,USD/mT,Global,Global

=== Rice date range & row count ===
+----------+----------+----+
|  min_date|  max_date|   n|
+----------+----------+----+
|2019-01-02|2026-08-14|1924|
+----------+----------+----+


=== Rice sample rows ===
Commodity,Code,Date,Unit,Open,High,Low,Price,Country,Origin,Name
Rice,RR1:COM,12/31/2025,USD/mT,211.34925,211.56975000000003,210.57750000000001,211.56975000000003,Global,Global,
Rice,RR1:COM,12/31/2024,USD/mT,307.48725,309.25125,307.48725,309.25125,Global,Global,
Rice,RR1:COM,12/31/2021,USD/mT,327.11175000000003,327.88349999999997,326.00925,327.55275,Global,Global,
Rice,RR1:COM,12/31/2020,USD/mT,269.892,269.892,269.892,269.892,Global,Global,
Rice,RR1:COM,12/31/2019,USD/mT,285.76800000000003,289.62675,285.76800000000003,289.62675,Global,Global,
Rice,RR1:COM,12/30/2025,USD/mT,207.9315,212.01075,207.60074999999998,209.03400000000002,Global,Global,
Rice,RR1:COM,12/30/2024,USD/mT,3

**_<u>Section 1:  Load tables</u>_**

In [3]:
# ── Load all source tables ─────────────────────────────────────────
df_wasde    = spark.table("srm.gld_stocks_to_use_ratio").toPandas()
df_wasde["commodity"] = df_wasde["commodity"].str.strip()
df_wasde = df_wasde[df_wasde["commodity"] != "Soybean Meal"].copy()
df_comtrade = spark.table("srm.comtrade_monthly_trade_world").toPandas()
df_bdi      = spark.table("srm.gld_bdi").toPandas()
df_policy   = spark.table("srm.gld_policy_ban_list").toPandas()
df_ksa      = spark.table("srm.gld_ksa_import_export").toPandas()
df_psd      = spark.table("srm.psd_alldata").toPandas()
df_price_intl = spark.table("dbo.gld_international_prices").toPandas()
df_price_te   = spark.table("dbo.gld_te_commodity_prices").toPandas()

COMMODITIES = ["Wheat","Corn","Rice","Soybean","Barley"]

print("=== Tables loaded ===")
print(f"WASDE:    {df_wasde.shape}")
print(f"Comtrade: {df_comtrade.shape}")
print(f"BDI:      {df_bdi.shape}")
print(f"Policy:   {df_policy.shape}")
print(f"KSA:      {df_ksa.shape}")
print(f"PSD:      {df_psd.shape}")
print(f"Price (Intl): {df_price_intl.shape}")
print(f"Price (TE):   {df_price_te.shape}")

# ── Save function ──────────────────────────────────────────────────
def save_to_lakehouse(df_pandas, table_name, schema="srm"):
    """
    Converts pandas DataFrame to Spark and saves
    to Fabric Lakehouse as Delta table.
    """
    full_name = f"{schema}.{table_name}"
    spark.createDataFrame(df_pandas) \
         .write \
         .mode("overwrite") \
         .option("overwriteSchema","true") \
         .format("delta") \
         .saveAsTable(full_name)
    count = spark.table(full_name).count()
    print(f"✓ {full_name}: {count} rows saved")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 5, Finished, Available, Finished, False)

=== Tables loaded ===
WASDE:    (338, 11)
Comtrade: (51557, 13)
BDI:      (92, 6)
Policy:   (113, 27)
KSA:      (321611, 16)
PSD:      (2088504, 12)
Price (Intl): (170856, 12)
Price (TE):   (15950, 11)


In [4]:
print(f"{'Commodity':<10} {'Threshold':>10} {'Mean':>8} {'Median':>8} {'Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 62)

STRESS_THRESHOLDS = {"Wheat": 26, "Corn": 21, "Rice": 30, "Soybean": 20, "Barley": 12}

for commodity in COMMODITIES:
    dc = df_wasde[df_wasde["commodity"] == commodity]["stocks_to_use_ratio"]
    print(f"{commodity:<10} {STRESS_THRESHOLDS[commodity]:>10} "
          f"{dc.mean():>8.2f} {dc.median():>8.2f} {dc.std():>8.2f} "
          f"{dc.min():>8.2f} {dc.max():>8.2f}")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 6, Finished, Available, Finished, False)

Commodity   Threshold     Mean   Median      Std      Min      Max
--------------------------------------------------------------
Wheat              26    26.87    26.53     1.64    24.90    32.85
Corn               21    21.17    21.68     1.41    17.99    23.03
Rice               30    30.97    30.91     1.29    28.80    33.88
Soybean            20    19.64    20.08     1.98    15.45    23.05
Barley             12    12.54    12.37     0.94    11.10    15.39


**_<u>Section 2:  Standardise + MAJOR_EXPORTERS + augmentation</u>_**

In [5]:
# ══════════════════════════════════════════════════════════════════
# SECTION 2: STANDARDISE ALL TABLES + DERIVE MAJOR EXPORTERS
# ══════════════════════════════════════════════════════════════════

# ── WASDE ──────────────────────────────────────────────────────────
if "year_month" not in df_wasde.columns:
    df_wasde["year_month"] = pd.to_datetime(df_wasde["period"].astype(str), format="%Y%m")
else:
    df_wasde["year_month"] = pd.to_datetime(df_wasde["year_month"])
df_wasde["domestic_feed"] = df_wasde["domestic_feed"].fillna(0)
df_wasde = df_wasde[df_wasde["commodity"].isin(COMMODITIES)].sort_values(["commodity","year_month"]).reset_index(drop=True)

# ── BDI ────────────────────────────────────────────────────────────
# 1. Convert date to period YYYYMM to match other tables
df_bdi["period"] = pd.to_datetime(
    df_bdi["date"]
).dt.strftime("%Y%m")

# 2. Keep only what you need — drop redundant columns
df_bdi = df_bdi[["period", "close"]].rename(
    columns={"close": "bdi_monthly_avg"}
)
# df_bdi["year_month"] = pd.to_datetime(df_bdi["period"])
df_bdi["year_month"] = pd.to_datetime(
    df_bdi["period"].astype(str), format="%Y%m", errors="coerce"
).dt.to_period("M").dt.to_timestamp()
df_bdi = df_bdi[["year_month","bdi_monthly_avg"]].sort_values("year_month").reset_index(drop=True)

# ── Comtrade ───────────────────────────────────────────────────────
if "year_month" not in df_comtrade.columns:
    df_comtrade["year_month"] = pd.to_datetime(df_comtrade["period"].astype(str), format="%Y%m")
else:
    df_comtrade["year_month"] = pd.to_datetime(df_comtrade["year_month"])
if "qty_mt" not in df_comtrade.columns:
    df_comtrade["qty_mt"] = df_comtrade["qty"] / 1000
df_comtrade = df_comtrade[(df_comtrade["qtyUnitCode"] != -1) & (df_comtrade["qty"] > 0)].copy()
df_comtrade = df_comtrade[~df_comtrade["reporterDesc"].isin(["Saudi Arabia","Saudi Arab"])].copy()

# ── Policy ─────────────────────────────────────────────────────────
df_policy["start_date"]        = pd.to_datetime(df_policy["start_date"],        errors="coerce")
df_policy["end_date"]          = pd.to_datetime(df_policy["end_date"],          errors="coerce")
df_policy["announcement_date"] = pd.to_datetime(df_policy["announcement_date"], errors="coerce")

# ── KSA ────────────────────────────────────────────────────────────
df_ksa["year"] = df_ksa["year"].astype(int)
df_ksa = df_ksa[df_ksa["trade_type"].isin(["Import","Imports"])]



# ── Policy augmentation — add missing critical events ──────────────
MISSING_POLICIES = [
    {
        "policy_id": 9001, "policy_type": "Ban", "status": "Inactive",
        "start_date": "2022-03-15", "end_date": "2022-07-01",
        "announcement_date": "2022-03-14", "issuing_country": "Russia",
        "commodity": "Wheat",
        "description": "Russia imposed a temporary ban on wheat exports following the invasion of Ukraine to protect domestic food supplies and control inflation. The ban applied to all wheat exports outside the Eurasian Economic Union.",
        "source_type": "Manual — USDA GAIN Russia Grain report Apr 2022"
    },
    {
        "policy_id": 9002, "policy_type": "Quota", "status": "Inactive",
        "start_date": "2022-07-01", "end_date": "2023-06-30",
        "announcement_date": "2022-06-25", "issuing_country": "Russia",
        "commodity": "Wheat",
        "description": "Russia replaced the export ban with a wheat export quota system limiting total wheat exports to protect domestic food supply following the Ukraine war disruption",
        "source_type": "Manual — USDA GAIN Russia Grain report Jul 2022"
    },
    {
        "policy_id": 9003, "policy_type": "Ban", "status": "Inactive",
        "start_date": "2022-05-13", "end_date": "2023-12-31",
        "announcement_date": "2022-05-13", "issuing_country": "India",
        "commodity": "Wheat",
        "description": "India imposed a sudden overnight ban on wheat exports with immediate effect following a severe heatwave that damaged domestic crop yields. The ban was introduced without prior notice to protect domestic food security and control rising prices.",
        "source_type": "Manual — USDA GAIN India Grain report May 2022"
    },
    {
        "policy_id": 9004, "policy_type": "Licensing", "status": "Inactive",
        "start_date": "2022-06-01", "end_date": "2023-06-30",
        "announcement_date": "2022-05-25", "issuing_country": "Ukraine",
        "commodity": "Wheat",
        "description": "Ukraine introduced export licensing requirements for wheat to manage domestic food security during the war with Russia. Export capacity severely disrupted by port blockades and infrastructure damage.",
        "source_type": "Manual — USDA GAIN Ukraine Grain report Jun 2022"
    },
    {
        "policy_id": 9005, "policy_type": "Ban", "status": "Inactive",
        "start_date": "2023-07-20", "end_date": "2024-03-31",
        "announcement_date": "2023-07-20", "issuing_country": "India",
        "commodity": "Rice",
        "description": "India banned exports of non-basmati white rice with immediate effect to control domestic rice prices after below average monsoon rainfall reduced planting areas significantly.",
        "source_type": "Manual — USDA GAIN India Grain report Aug 2023"
    },
    {
        "policy_id": 9006, "policy_type": "Tax", "status": "Inactive",
        "start_date": "2023-08-25", "end_date": "2024-03-31",
        "announcement_date": "2023-08-24", "issuing_country": "India",
        "commodity": "Rice",
        "description": "India imposed 20 percent export duty on parboiled rice exports to augment domestic supplies and calm local prices. Unmilled rice and husked brown rice also attracted 20 percent export levy.",
        "source_type": "Manual — USDA GAIN India Grain report Sep 2023"
    },
    {
        "policy_id": 9007, "policy_type": "Ban", "status": "Inactive",
        "start_date": "2022-09-08", "end_date": "2023-12-31",
        "announcement_date": "2022-09-08", "issuing_country": "India",
        "commodity": "Rice",
        "description": "India banned exports of broken rice and imposed 20 percent duty on various grades of rice as the worlds biggest rice exporter tried to augment supplies and calm local prices after below average monsoon rainfall curtailed planting.",
        "source_type": "Manual — USDA GAIN India Grain report Sep 2022"
    }
]

df_missing = pd.DataFrame(MISSING_POLICIES)

# Fix datetime dtype to match policy table
date_cols = ["start_date","end_date","announcement_date"]
for col in date_cols:
    df_missing[col] = pd.to_datetime(df_missing[col]).astype("datetime64[us]")

# Align columns
for col in df_policy.columns:
    if col not in df_missing.columns:
        if pd.api.types.is_datetime64_any_dtype(df_policy[col]):
            df_missing[col] = pd.NaT
            df_missing[col] = df_missing[col].astype("datetime64[us]")
        elif pd.api.types.is_float_dtype(df_policy[col]):
            df_missing[col] = np.nan
        elif pd.api.types.is_integer_dtype(df_policy[col]):
            df_missing[col] = np.nan
        else:
            df_missing[col] = None

df_missing = df_missing[df_policy.columns.tolist()]

# Concat
df_policy = pd.concat(
    [df_policy.reset_index(drop=True), df_missing.reset_index(drop=True)],
    ignore_index=True, sort=False
)

df_policy = df_policy.drop_duplicates(subset=["policy_id"]).reset_index(drop=True)
print(f"After deduplication: {len(df_policy)} rows")

print(f"Policy table augmented: {len(df_policy)} rows (7 critical events added)")

# Save augmented policy table
save_to_lakehouse(df_policy, "gld_policy_ban_list_augmented")

# ── Master monthly spine ───────────────────────────────────────────
date_min = df_wasde["year_month"].min()
date_max = df_wasde["year_month"].max()

TRAINING_CUTOFF = pd.Timestamp("2025-12-01")
SCORING_CUTOFF  = pd.Timestamp("2026-06-01")

spine = pd.MultiIndex.from_product(
    [pd.date_range(date_min, date_max, freq="MS"), COMMODITIES],
    names=["year_month","commodity"]
).to_frame(index=False)

spine_train = spine[spine["year_month"] <= TRAINING_CUTOFF].copy()
spine_score = spine[
    (spine["year_month"] > TRAINING_CUTOFF) &
    (spine["year_month"] <= SCORING_CUTOFF)
].copy()

# ── NEW — dedicated spine for Policy/KSA, extends through the actual
# current scoring month. spine_train stops at TRAINING_CUTOFF on purpose
# (Prophet's baseline training window) — Policy and KSA need to reach
# further forward than that, so they get their own spine here.
SCORING_END = pd.Timestamp("2026-07-01")

spine_full = pd.MultiIndex.from_product(
    [pd.date_range(date_min, SCORING_END, freq="MS"), COMMODITIES],
    names=["year_month","commodity"]
).to_frame(index=False)

print(f"Training spine:  {spine_train.shape}")
print(f"  {spine_train['year_month'].min().date()} → {spine_train['year_month'].max().date()}")
print(f"Scoring spine:   {spine_score.shape}")
print(f"  {spine_score['year_month'].min().date()} → {spine_score['year_month'].max().date()}")
print(f"Forecast target: Jul 2026, Aug 2026, Sep 2026")

# ══════════════════════════════════════════════════════════════════
# DERIVE MAJOR EXPORTERS FROM USDA PSD DATA
# Fully integrated — no separate cell needed
# ══════════════════════════════════════════════════════════════════

PSD_COMMODITY_MAP = {
    "Wheat":        "Wheat",
    "Corn":         "Corn",
    "Rice, Milled": "Rice",
    "Oilseed, Soybean": "Soybean",
    "Barley":       "Barley"
}

EXCLUDE_COUNTRIES = [
    "World","European Union","Former Soviet Union",
    "Africa","Asia","Europe","North America",
    "South America","Oceania","Central America",
    "Middle East","Southeast Asia","South Asia",
    "Sub-Saharan Africa","North Africa","East Asia",
    "Central Asia","Caribbean","Baltic States","Other"
]

# Filter PSD to exports only for your commodities
df_exports_psd = df_psd[
    (df_psd["Commodity_Description"].isin(PSD_COMMODITY_MAP.keys())) &
    (df_psd["Attribute_Description"] == "Exports") &
    (df_psd["Market_Year"] >= 2019) &
    (df_psd["Market_Year"] <= 2024) &
    (df_psd["Value"] > 0) &
    (~df_psd["Country_Name"].isin(EXCLUDE_COUNTRIES))
].copy()

df_exports_psd["commodity"] = df_exports_psd["Commodity_Description"].map(PSD_COMMODITY_MAP)

# Average exports 2019-2024 per country
avg_exports = df_exports_psd.groupby(["Country_Name","commodity"])["Value"].mean().reset_index()
avg_exports.columns = ["country","commodity","avg_export_1000mt"]
avg_exports["avg_export_mt"] = avg_exports["avg_export_1000mt"] * 1000

# World total per commodity
world_total = avg_exports.groupby("commodity")["avg_export_mt"].sum().reset_index()
world_total.columns = ["commodity","world_total_mt"]
avg_exports = avg_exports.merge(world_total, on="commodity")
avg_exports["export_share_pct"] = (avg_exports["avg_export_mt"] / avg_exports["world_total_mt"] * 100).round(3)

# Apply 2% threshold
MAJOR_EXPORTER_THRESHOLD = 2.0

MAJOR_EXPORTERS = {}
for commodity in COMMODITIES:
    major = avg_exports[
        (avg_exports["commodity"] == commodity) &
        (avg_exports["export_share_pct"] >= MAJOR_EXPORTER_THRESHOLD)
    ].sort_values("export_share_pct", ascending=False)
    MAJOR_EXPORTERS[commodity] = major["country"].tolist()

print("\n=== Major exporters derived from USDA PSD (>=2% world share) ===")
for commodity, countries in MAJOR_EXPORTERS.items():
    print(f"\n{commodity} ({len(countries)} countries):")
    top = avg_exports[
        (avg_exports["commodity"] == commodity) &
        (avg_exports["export_share_pct"] >= MAJOR_EXPORTER_THRESHOLD)
    ].sort_values("export_share_pct", ascending=False)[["country","export_share_pct"]]
    print(top.to_string(index=False))

# Save major exporters reference
save_to_lakehouse(
    avg_exports[avg_exports["export_share_pct"] >= MAJOR_EXPORTER_THRESHOLD],
    "major_exporters_reference"
)

print("\n=== Section 2 complete ===")
print(f"WASDE:          {df_wasde.shape}")
print(f"BDI:            {df_bdi.shape}")
print(f"Comtrade:       {df_comtrade.shape}")
print(f"Policy:         {df_policy.shape} (augmented)")
print(f"KSA:            {df_ksa.shape}")
print(f"Full spine:     {spine.shape}")
print(f"Training spine: {spine_train.shape}")
print(f"MAJOR_EXPORTERS: {len(MAJOR_EXPORTERS)} commodities")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 7, Finished, Available, Finished, False)

After deduplication: 120 rows
Policy table augmented: 120 rows (7 critical events added)
✓ srm.gld_policy_ban_list_augmented: 120 rows saved
Training spine:  (360, 2)
  2020-01-01 → 2025-12-01
Scoring spine:   (30, 2)
  2026-01-01 → 2026-06-01
Forecast target: Jul 2026, Aug 2026, Sep 2026

=== Major exporters derived from USDA PSD (>=2% world share) ===

Wheat (8 countries):
      country  export_share_pct
       Russia            24.194
       Canada            13.860
United States            13.038
    Australia            12.874
      Ukraine            10.259
    Argentina             6.214
   Kazakhstan             4.949
       Turkey             4.157

Corn (5 countries):
      country  export_share_pct
United States            31.858
       Brazil            21.750
    Argentina            18.438
      Ukraine            14.243
       Russia             2.510

Rice (9 countries):
      country  export_share_pct
        India            34.588
     Thailand            14.223
    

In [6]:
# # Check null percentage for every policy column
# null_pct = (
#     df_policy.isnull().sum() / len(df_policy) * 100
# ).round(1).sort_values(ascending=False)

# print("=== Policy table NULL % per column ===")
# print(null_pct.to_string())

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 8, Finished, Available, Finished, False)

In [7]:
# # See what description contains
# print("Sample descriptions:")
# print(
#     df_policy[["policy_type","commodity",
#                "issuing_country","description"]]
#     .head(15)
#     .to_string(index=False)
# )

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 9, Finished, Available, Finished, False)

**_<u>Section 3:  WASDE (STU and PPI) features (includes gap fix)</u>_**

In [8]:
# ══════════════════════════════════════════════════════════════════
# SECTION 3: WASDE FEATURES
# Covers:
# - Global STU ratio (indicator 1 — weight 20%)
# - Production Potential Index (indicator 2 — weight 12%)
# Includes: gap fill for missing months (Oct 2023, Oct 2025)
# ══════════════════════════════════════════════════════════════════

def build_wasde_features(df_wasde):
    """
    Builds all features from WASDE table.
    Covers STU ratio and Production Potential Index.
    """
    results = []

    for commodity in COMMODITIES:
        dc = df_wasde[
            df_wasde["commodity"] == commodity
        ].copy().sort_values("year_month").reset_index(drop=True)

        print(f"\n{commodity}: {len(dc)} months")

        # ══════════════════════════════════════════
        # INDICATOR 1: Global STU ratio
        # ══════════════════════════════════════════

        dc["stu_ratio"]           = dc["stocks_to_use_ratio"]
        dc["stu_mom_change"]      = dc["stu_ratio"].diff()
        dc["stu_3m_avg"]          = dc["stu_ratio"].rolling(3).mean()
        dc["stu_6m_avg"]          = dc["stu_ratio"].rolling(6).mean()
        dc["stu_yoy_change"]      = dc["stu_ratio"].diff(12)

        STRESS_THRESHOLDS = {"Wheat": 26, "Corn": 21, "Rice": 30, "Soybean": 20, "Barley": 12}
        threshold = STRESS_THRESHOLDS[commodity]

        dc["stu_stress_flag"]     = (dc["stu_ratio"] < threshold).astype(int)
        dc["stu_sustained_stress"]= dc["stu_stress_flag"].rolling(3).sum()

        # ══════════════════════════════════════════
        # INDICATOR 2: Production Potential Index
        # ══════════════════════════════════════════

        # 12 month rolling average — enough history with 66 months
        dc["prod_12m_avg"]        = dc["production"].shift(1).rolling(12).mean()
        dc["ppi_base"]            = dc["production"] / dc["prod_12m_avg"] * 100
        dc["prod_mom_revision"]   = dc["production"].diff()
        dc["prod_rev_pct"]        = dc["production"].pct_change() * 100
        dc["prod_cut_flag"]       = (dc["prod_mom_revision"] < 0).astype(int)
        dc["consecutive_prod_cuts"]= dc["prod_cut_flag"].rolling(3).sum()
        dc["ppi_3m_trend"]        = dc["ppi_base"].diff(3)

        def compute_ppi_score(row):
            score = 0
            base = row["ppi_base"]
            if pd.isna(base):
                return np.nan
            if base >= 103:    score += 0
            elif base >= 100:  score += 15
            elif base >= 97:   score += 30
            elif base >= 94:   score += 45
            else:              score += 60

            trend = row["ppi_3m_trend"]
            if pd.isna(trend):     score += 10
            elif trend < -3:       score += 25
            elif trend < 0:        score += 10
            else:                  score += 0

            cuts = row["consecutive_prod_cuts"]
            if pd.isna(cuts):      score += 0
            else:                  score += min(cuts * 5, 15)

            return min(score, 100)

        dc["ppi_score"]           = dc.apply(compute_ppi_score, axis=1)

        # ══════════════════════════════════════════
        # Supporting stock features
        # ══════════════════════════════════════════

        dc["stocks_mom_change"]   = dc["ending_stocks"].diff()
        dc["stocks_3m_trend"]     = dc["ending_stocks"].pct_change(3) * 100
        dc["stocks_6m_avg"]       = dc["ending_stocks"].rolling(6).mean()
        dc["stocks_drawdown_flag"]= (dc["ending_stocks"] < dc["stocks_6m_avg"]).astype(int)
        dc["months_of_cover"]     = dc["ending_stocks"] / dc["total_use"] * 12
        dc["export_to_prod_ratio"]= dc["exports"] / dc["production"]

        results.append(dc)

    df_out = pd.concat(results, ignore_index=True)

    wasde_cols = [
        "year_month","commodity",
        "stu_ratio","stu_mom_change","stu_3m_avg",
        "stu_6m_avg","stu_yoy_change",
        "stu_stress_flag","stu_sustained_stress",
        "ppi_base","ppi_score","ppi_3m_trend",
        "prod_mom_revision","prod_rev_pct",
        "prod_cut_flag","consecutive_prod_cuts",
        "stocks_mom_change","stocks_3m_trend",
        "stocks_drawdown_flag","months_of_cover",
        "export_to_prod_ratio"
    ]

    return df_out[wasde_cols]


# ── Run ────────────────────────────────────────────────────────────
df_wasde_features = build_wasde_features(df_wasde)

# ── Gap fill — missing months (Oct 2023, Oct 2025) ────────────────
print("\n=== Checking WASDE gaps ===")
for commodity in COMMODITIES:
    dc = df_wasde_features[
        df_wasde_features["commodity"]==commodity
    ]["year_month"].sort_values()
    full_range = pd.date_range(dc.min(), dc.max(), freq="MS")
    missing = full_range[~full_range.isin(dc)]
    if len(missing) > 0:
        print(f"{commodity} missing: {[str(m.date()) for m in missing]}")
    else:
        print(f"{commodity}: no gaps ✓")

# Create complete monthly spine
wasde_spine = pd.MultiIndex.from_product(
    [pd.date_range(df_wasde_features["year_month"].min(), df_wasde_features["year_month"].max(), freq="MS"), COMMODITIES],
    names=["year_month","commodity"]
).to_frame(index=False)

# Merge and forward fill
df_wasde_features = wasde_spine.merge(df_wasde_features, on=["year_month","commodity"], how="left")
df_wasde_features = df_wasde_features.sort_values(["commodity","year_month"]).reset_index(drop=True)
numeric_cols = df_wasde_features.select_dtypes(include="number").columns.tolist()
df_wasde_features[numeric_cols] = df_wasde_features.groupby("commodity")[numeric_cols].ffill()

print("\n=== WASDE gaps after fix ===")
for commodity in COMMODITIES:
    dc = df_wasde_features[
        df_wasde_features["commodity"]==commodity
    ]["year_month"].sort_values()
    full_range = pd.date_range(dc.min(), dc.max(), freq="MS")
    missing = full_range[~full_range.isin(dc)]
    if len(missing) > 0:
        print(f"{commodity} still missing: {[str(m.date()) for m in missing]}")
    else:
        print(f"{commodity}: no gaps ✓")

# ── Save ───────────────────────────────────────────────────────────
save_to_lakehouse(df_wasde_features, "features_wasde")

print(f"\nWASDE features saved: {df_wasde_features.shape}")
print("\nSample output:")
print(df_wasde_features[[
    "year_month","commodity",
    "stu_ratio","stu_stress_flag",
    "ppi_base","ppi_score",
    "consecutive_prod_cuts"
]].tail(9).to_string(index=False))

# ── Sanity check ───────────────────────────────────────────────────
print("\n=== Sanity Check: Ukraine War Wheat (2022) ===")
print(df_wasde_features[
    (df_wasde_features["commodity"]=="Wheat") &
    (df_wasde_features["year_month"]>="2022-01-01") &
    (df_wasde_features["year_month"]<="2022-06-01")
][[
    "year_month","stu_ratio","ppi_score",
    "consecutive_prod_cuts","months_of_cover"
]].to_string(index=False))

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 10, Finished, Available, Finished, False)


Wheat: 65 months

Corn: 65 months

Rice: 65 months

Soybean: 65 months

Barley: 78 months

=== Checking WASDE gaps ===
Wheat missing: ['2023-10-01', '2025-10-01']
Corn missing: ['2023-10-01', '2025-10-01']
Rice missing: ['2023-10-01', '2025-10-01']
Soybean missing: ['2023-10-01', '2025-10-01']
Barley missing: ['2025-10-01']

=== WASDE gaps after fix ===
Wheat: no gaps ✓
Corn: no gaps ✓
Rice: no gaps ✓
Soybean: no gaps ✓
Barley: no gaps ✓
✓ srm.features_wasde: 395 rows saved

WASDE features saved: (395, 21)

Sample output:
year_month commodity  stu_ratio  stu_stress_flag   ppi_base  ppi_score  consecutive_prod_cuts
2025-11-01     Wheat      26.20              0.0 103.485881        5.0                    1.0
2025-12-01     Wheat      26.39              0.0 104.222076        0.0                    0.0
2026-01-01     Wheat      26.66              0.0 104.298665        0.0                    0.0
2026-02-01     Wheat      26.53              0.0 103.725944        5.0                    1.0
2

In [9]:
save_to_lakehouse(df_wasde_features, "features_wasde")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 11, Finished, Available, Finished, False)

✓ srm.features_wasde: 395 rows saved


In [10]:
print(df_wasde.columns.tolist())

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 12, Finished, Available, Finished, False)

['commodity', 'beginning_stocks', 'production', 'imports', 'domestic_feed', 'total_domestic', 'exports', 'ending_stocks', 'period', 'total_use', 'stocks_to_use_ratio', 'year_month']


In [11]:
# # ── Build ppi_base manually from raw production, same formula as the pipeline ─
# dc_wheat = df_wasde[df_wasde["commodity"]=="Wheat"][["year_month","production"]].copy()
# dc_wheat["year_month"] = pd.to_datetime(dc_wheat["year_month"])
# dc_wheat = dc_wheat.sort_values("year_month").reset_index(drop=True)

# dc_wheat["rolling_12m_avg"] = dc_wheat["production"].rolling(window=12, min_periods=6).mean()
# dc_wheat["ppi_base"] = (dc_wheat["production"] / dc_wheat["rolling_12m_avg"]) * 100

# print("Last 12 months of Wheat ppi_base:")
# print(dc_wheat[["year_month","production","ppi_base"]].tail(12).to_string(index=False))

# print("\nAugust values historically (check for seasonal pattern):")
# aug_values = dc_wheat[dc_wheat["year_month"].dt.month == 8]
# print(aug_values[["year_month","production","ppi_base"]].to_string(index=False))

# print("\nMonth-over-month change stats (is a ~3pt jump normal?):")
# dc_wheat["mom_change"] = dc_wheat["ppi_base"].diff()
# print(dc_wheat["mom_change"].describe())

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 13, Finished, Available, Finished, False)

**_<u>Section 4 — BDI Features</u>_**

In [12]:
def build_bdi_features(df_bdi):
    """
    Builds all freight and logistics features from BDI table.
    Covers: BDI Index (indicator 7 — weight 19%)
    
    BDI = Baltic Dry Index
    Measures cost of shipping dry bulk commodities
    (grain, coal, iron ore) across major sea routes.
    
    Rising BDI = higher freight costs = supply stress
    Falling BDI = cheaper freight = easier supply flow
    """
    dc = df_bdi.copy().sort_values("year_month").reset_index(drop=True)

    # ── Raw BDI level ──────────────────────────────────────────────
    dc["bdi"] = dc["bdi_monthly_avg"]

    # ── MoM change ────────────────────────────────────────────────
    # Absolute change from last month
    dc["bdi_mom_change"] = dc["bdi"].diff()

    # Percentage change from last month
    dc["bdi_mom_pct"] = dc["bdi"].pct_change() * 100

    # ── Rolling averages ──────────────────────────────────────────
    # Short term trend
    dc["bdi_3m_avg"]  = dc["bdi"].rolling(3).mean()

    # Medium term trend
    dc["bdi_6m_avg"]  = dc["bdi"].rolling(6).mean()

    # Long term baseline
    dc["bdi_12m_avg"] = dc["bdi"].rolling(12).mean()

    # ── Year on year change ────────────────────────────────────────
    # Removes seasonal effects
    # Positive = freight more expensive than same month last year
    dc["bdi_yoy_change"] = dc["bdi"].diff(12)
    dc["bdi_yoy_pct"]    = dc["bdi"].pct_change(12) * 100

    # ── Z-score ───────────────────────────────────────────────────
    # How extreme is current BDI vs recent 12m history?
    # Above +1.5 = unusually high freight costs = stress signal
    # Below -1.5 = unusually low = slack demand signal
    dc["bdi_12m_std"] = dc["bdi"].rolling(12).std()
    dc["bdi_z_score"] = (
        (dc["bdi"] - dc["bdi_12m_avg"]) / dc["bdi_12m_std"]
    )

    # ── BDI vs rolling averages ───────────────────────────────────
    # Is current BDI above or below recent trend?
    # Above 1.0 = elevated vs recent trend
    dc["bdi_vs_3m_avg"]  = dc["bdi"] / dc["bdi_3m_avg"]
    dc["bdi_vs_6m_avg"]  = dc["bdi"] / dc["bdi_6m_avg"]
    dc["bdi_vs_12m_avg"] = dc["bdi"] / dc["bdi_12m_avg"]

    # ── Spike flag ────────────────────────────────────────────────
    # Binary flag when BDI is more than 1.5 std above 12m average
    # Indicates abnormal freight cost pressure
    dc["bdi_spike_flag"] = (
        dc["bdi_z_score"] > 1.5
    ).astype(int)

    # ── Sustained high BDI ────────────────────────────────────────
    # 3 consecutive months elevated = structural freight pressure
    # not just a one-month spike
    dc["bdi_sustained_high"] = (
        dc["bdi_spike_flag"].rolling(3).sum()
    )

    # ── Direction flags ───────────────────────────────────────────
    # Is BDI rising or falling this month?
    dc["bdi_rising_flag"] = (
        dc["bdi_mom_change"] > 0
    ).astype(int)

    # How many of last 3 months was BDI rising?
    # 3 = rising all 3 months = sustained upward pressure
    # 0 = falling all 3 months = sustained relief
    dc["bdi_3m_direction"] = (
        dc["bdi_rising_flag"].rolling(3).sum()
    )

    # ── Volatility ────────────────────────────────────────────────
    # High BDI volatility = uncertain freight market
    # = harder to plan procurement
    dc["bdi_3m_volatility"] = (
        dc["bdi"].rolling(3).std() /
        dc["bdi"].rolling(3).mean() * 100
    )

    bdi_cols = [
        "year_month",
        "bdi",
        "bdi_mom_change",
        "bdi_mom_pct",
        "bdi_3m_avg",
        "bdi_6m_avg",
        "bdi_12m_avg",
        "bdi_yoy_change",
        "bdi_yoy_pct",
        "bdi_z_score",
        "bdi_vs_3m_avg",
        "bdi_vs_6m_avg",
        "bdi_vs_12m_avg",
        "bdi_spike_flag",
        "bdi_sustained_high",
        "bdi_rising_flag",
        "bdi_3m_direction",
        "bdi_3m_volatility"
    ]

    return dc[bdi_cols]


# Run
df_bdi_features = build_bdi_features(df_bdi)

# Save to Lakehouse
spark.createDataFrame(df_bdi_features).write \
     .format("delta").mode("overwrite") \
     .saveAsTable("Srm.features_bdi")

print(f"BDI features saved: {df_bdi_features.shape}")
print(f"\nDate range: {df_bdi_features['year_month'].min().date()} "
      f"→ {df_bdi_features['year_month'].max().date()}")

print("\nSample output:")
print(df_bdi_features[[
    "year_month","bdi","bdi_z_score",
    "bdi_spike_flag","bdi_3m_direction",
    "bdi_vs_3m_avg"
]].tail(6).to_string(index=False))

# ── Sanity checks ──────────────────────────────────────────────────
print("\n=== Sanity Check 1: BDI spike period (late 2021) ===")
print(df_bdi_features[
    (df_bdi_features["year_month"] >= "2021-09-01") &
    (df_bdi_features["year_month"] <= "2022-03-01")
][[
    "year_month","bdi","bdi_z_score",
    "bdi_spike_flag","bdi_sustained_high"
]].to_string(index=False))

print("\n=== Sanity Check 2: BDI low period (2023) ===")
print(df_bdi_features[
    (df_bdi_features["year_month"] >= "2023-01-01") &
    (df_bdi_features["year_month"] <= "2023-06-01")
][[
    "year_month","bdi","bdi_z_score",
    "bdi_spike_flag","bdi_3m_direction"
]].to_string(index=False))

print("\n=== BDI summary statistics ===")
print(df_bdi_features[[
    "bdi","bdi_z_score","bdi_mom_pct"
]].describe().round(2))

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 14, Finished, Available, Finished, False)

BDI features saved: (92, 18)

Date range: 2019-01-01 → 2026-08-01

Sample output:
year_month     bdi  bdi_z_score  bdi_spike_flag  bdi_3m_direction  bdi_vs_3m_avg
2026-03-01 2047.45     0.495940               0               2.0       1.046294
2026-04-01 2442.75     1.545059               1               3.0       1.122054
2026-05-01 3050.70     2.544548               1               3.0       1.213661
2026-06-01 2774.86     1.471086               0               2.0       1.006806
2026-07-01 2769.91     1.220879               0               1.0       0.966757
2026-08-01 2976.30     1.421370               0               1.0       1.047861

=== Sanity Check 1: BDI spike period (late 2021) ===
year_month     bdi  bdi_z_score  bdi_spike_flag  bdi_sustained_high
2021-09-01 4287.77     1.849206               1                 3.0
2021-10-01 4819.95     1.792589               1                 3.0
2021-11-01 2780.45    -0.015660               0                 2.0
2021-12-01 2832.11    -0.

In [13]:
save_to_lakehouse(df_bdi_features,   "features_bdi")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 15, Finished, Available, Finished, False)

✓ srm.features_bdi: 92 rows saved


**_<u>Section 5 — Demand Pressure Features (from Comtrade)</u>_**

In [14]:
def build_demand_features(df_comtrade):
    """
    Builds demand pressure features from Comtrade data.
    Covers:
    - Global import competition (indicator 4 — weight 6%)
    - Individual dominant buyer tracking (indicator 3 — weight 7%)

    Two level approach:
    Level 1: Collective pressure — all major buyers combined
    Level 2: Concentrated pressure — top N buyers individually
             Countries identified dynamically from data
             No hardcoded country names
    """

    COMMODITY_HS = {"Wheat": ["1001"], "Corn": ["1005"], "Rice": ["1006"], "Soybean": ["1201"], "Barley": ["1003"]}

    def map_commodity(cmd):
        cmd_str = str(cmd)
        for commodity, codes in COMMODITY_HS.items():
            if any(cmd_str.startswith(c) for c in codes):
                return commodity
        return None

    df_c = df_comtrade.copy()
    df_c["commodity"] = df_c["cmdCode"].apply(map_commodity)
    df_c = df_c[df_c["commodity"].isin(COMMODITIES)].copy()

    EXCLUDE = ["Saudi Arabia", "Saudi Arab"]
    df_c = df_c[~df_c["reporterDesc"].isin(EXCLUDE)].copy()
    df_c["year"] = df_c["year_month"].dt.year

    print(f"Comtrade rows after filtering: {len(df_c)}")
    print(f"Commodities: {df_c['commodity'].unique()}")
    print(f"Date range: {df_c['year_month'].min().date()} → {df_c['year_month'].max().date()}")

    results = []

    for commodity in COMMODITIES:
        dc = df_c[df_c["commodity"] == commodity].copy()

        if dc.empty:
            print(f"WARNING: No Comtrade data for {commodity}")
            continue

        print(f"\n{commodity}: {dc['reporterDesc'].nunique()} countries, {dc['year_month'].nunique()} months")

        # ── Step 1: Dynamic major buyers per year ─────────────────
        annual = dc.groupby(["year","reporterDesc"])["qty_mt"].sum().reset_index()

        world_total = annual.groupby("year")["qty_mt"].sum().reset_index()
        world_total.columns = ["year","world_total_mt"]

        annual = annual.merge(world_total, on="year")
        annual["import_share"] = annual["qty_mt"] / annual["world_total_mt"]
        annual["rank"] = annual.groupby("year")["qty_mt"].rank(ascending=False, method="dense")
        annual["is_major_buyer"] = ((annual["import_share"] >= 0.03) | (annual["rank"] <= 10)).astype(int)

        # ── Step 2: Identify individually tracked countries ────────
        top3_by_year = annual[annual["rank"] <= 3]
        top3_frequency = top3_by_year.groupby("reporterDesc")["year"].count().reset_index()
        top3_frequency.columns = ["country","years_in_top3"]
        total_years = annual["year"].nunique()

        INDIVIDUAL_TRACK_THRESHOLD = 0.50
        individually_tracked = top3_frequency[top3_frequency["years_in_top3"] >= total_years * INDIVIDUAL_TRACK_THRESHOLD]["country"].tolist()
        print(f"  Individually tracked for {commodity}: {individually_tracked}")

        # ── Step 3: Join major buyer flag to monthly ───────────────
        dc = dc.merge(annual[["year","reporterDesc","import_share","rank","is_major_buyer"]], on=["year","reporterDesc"], how="left")
        dc_major = dc[dc["is_major_buyer"] == 1].copy()

        # ── Step 4: Monthly aggregate — collective pressure ────────
        monthly_agg = dc_major.groupby("year_month").agg(driver_total_mt=("qty_mt","sum"), n_major_buyers=("reporterDesc","nunique")).reset_index().sort_values("year_month")
        monthly_agg["driver_12m_avg"] = monthly_agg["driver_total_mt"].rolling(12).mean()
        monthly_agg["demand_surge_ratio"] = monthly_agg["driver_total_mt"] / monthly_agg["driver_12m_avg"]
        monthly_agg["demand_mom_change"] = monthly_agg["driver_total_mt"].pct_change() * 100

        # ── Step 5: Individual country tracking ───────────────────
        individual_dfs = []

        for country in individually_tracked:
            country_data = dc[dc["reporterDesc"] == country].groupby("year_month")["qty_mt"].sum().reset_index()

            if country_data.empty:
                continue

            col_prefix = country.lower().replace(" ","_").replace(",","").replace(".","").replace("'","")
            country_data.columns = ["year_month", f"{col_prefix}_imports_mt"]
            country_data = country_data.sort_values("year_month")
            country_data[f"{col_prefix}_12m_avg"] = country_data[f"{col_prefix}_imports_mt"].rolling(12).mean()
            country_data[f"{col_prefix}_surge_ratio"] = country_data[f"{col_prefix}_imports_mt"] / country_data[f"{col_prefix}_12m_avg"]
            country_data[f"{col_prefix}_surge_flag"] = (country_data[f"{col_prefix}_surge_ratio"] > 1.25).astype(int)
            individual_dfs.append(country_data)

        # ── Step 6: Countries surging simultaneously ───────────────
        country_monthly = dc_major.groupby(["year_month","reporterDesc"])["qty_mt"].sum().reset_index()
        country_monthly = country_monthly.sort_values(["reporterDesc","year_month"])
        country_monthly["country_12m_avg"] = country_monthly.groupby("reporterDesc")["qty_mt"].transform(lambda x: x.rolling(12).mean())
        country_monthly["country_surge"] = (country_monthly["qty_mt"] / country_monthly["country_12m_avg"] > 1.20).astype(int)
        simultaneous = country_monthly.groupby("year_month")["country_surge"].sum().reset_index()
        simultaneous.columns = ["year_month","countries_surging"]

        # ── Step 7: Top buyer decomposition ───────────────────────
        top_buyer = dc_major[dc_major["rank"] == 1].groupby("year_month").agg(top_buyer_mt=("qty_mt","sum"), top_buyer_country=("reporterDesc","first")).reset_index().sort_values("year_month")
        top_buyer["top_buyer_12m_avg"] = top_buyer["top_buyer_mt"].rolling(12).mean()
        top_buyer["top_buyer_surge_ratio"] = top_buyer["top_buyer_mt"] / top_buyer["top_buyer_12m_avg"]
        top_buyer["top_buyer_surging"] = (top_buyer["top_buyer_surge_ratio"] > 1.25).astype(int)

        top_buyer_share = dc_major.groupby(["year_month","rank"])["qty_mt"].sum().reset_index()
        top1_share = top_buyer_share[top_buyer_share["rank"] == 1][["year_month","qty_mt"]].rename(columns={"qty_mt":"top1_mt"})
        monthly_total = dc_major.groupby("year_month")["qty_mt"].sum().reset_index()
        monthly_total.columns = ["year_month","total_mt"]
        top1_share = top1_share.merge(monthly_total, on="year_month")
        top1_share["top_buyer_share_of_total"] = top1_share["top1_mt"] / top1_share["total_mt"]
        top_buyer = top_buyer.merge(top1_share[["year_month","top_buyer_share_of_total"]], on="year_month", how="left")
        top_buyer["concentrated_surge_flag"] = ((top_buyer["top_buyer_surge_ratio"] > 1.25) & (top_buyer["top_buyer_share_of_total"] > 0.40)).astype(int)

        # ── Step 8: Merge all features ─────────────────────────────
        demand = monthly_agg.merge(simultaneous, on="year_month", how="left").merge(top_buyer[["year_month","top_buyer_country","top_buyer_surge_ratio","top_buyer_surging","top_buyer_share_of_total","concentrated_surge_flag"]], on="year_month", how="left")

        for country_df in individual_dfs:
            demand = demand.merge(country_df, on="year_month", how="left")

        demand["countries_surging"] = demand["countries_surging"].fillna(0)
        demand["top_buyer_surging"] = demand["top_buyer_surging"].fillna(0)
        demand["top_buyer_surge_ratio"] = demand["top_buyer_surge_ratio"].fillna(1.0)
        demand["concentrated_surge_flag"] = demand["concentrated_surge_flag"].fillna(0)

        # ── Step 9: Composite demand pressure score (0-100) ───────
        def demand_score(row):
            score = 0

            surge = row.get("demand_surge_ratio", 1.0)
            if pd.isna(surge):      score += 0
            elif surge >= 1.40:     score += 40
            elif surge >= 1.25:     score += 28
            elif surge >= 1.10:     score += 16
            elif surge >= 1.00:     score += 5
            else:                   score += 0

            top = row.get("top_buyer_surge_ratio", 1.0)
            conc = row.get("concentrated_surge_flag", 0)
            if pd.isna(top):        score += 0
            elif conc == 1:         score += 30
            elif top >= 1.25:       score += 15
            else:                   score += 0

            n = row.get("countries_surging", 0)
            if pd.isna(n):          score += 0
            elif n >= 5:            score += 30
            elif n >= 4:            score += 22
            elif n >= 3:            score += 15
            elif n >= 2:            score += 7
            else:                   score += 0

            return min(score, 100)

        demand["demand_pressure_score"] = demand.apply(demand_score, axis=1)
        demand["commodity"] = commodity
        demand["individually_tracked_countries"] = str(individually_tracked)
        results.append(demand)

    if not results:
        print("ERROR: No demand features built")
        return pd.DataFrame()

    df_out = pd.concat(results, ignore_index=True)
    return df_out


# ── Run build function ─────────────────────────────────────────────
df_demand_features = build_demand_features(df_comtrade)

# ── Data completeness flag ─────────────────────────────────────────
reporting_counts = df_comtrade[df_comtrade["cmdCode"].astype(str).str.startswith(("1001","1005","1006","1201","1003"))].groupby(["year_month","cmdCode"])["reporterDesc"].nunique().reset_index()
reporting_counts.columns = ["year_month","cmdCode","countries_reporting"]

cmd_map = {"1001":"Wheat","1005":"Corn","1006":"Rice","1201":"Soybean","1003":"Barley"}
reporting_counts["commodity"] = reporting_counts["cmdCode"].astype(str).str[:4].map(cmd_map)

normal_counts = reporting_counts.groupby("commodity")["countries_reporting"].median().reset_index()
normal_counts.columns = ["commodity","normal_reporting_count"]

print("Normal reporting counts (median):")
print(normal_counts.to_string(index=False))

reporting_counts = reporting_counts.merge(normal_counts, on="commodity")
reporting_counts["completeness_ratio"] = reporting_counts["countries_reporting"] / reporting_counts["normal_reporting_count"]
reporting_counts["demand_data_completeness"] = reporting_counts["completeness_ratio"].clip(0, 1).round(3)

COMPLETENESS_THRESHOLD = 0.50
reporting_counts["demand_data_incomplete"] = (reporting_counts["demand_data_completeness"] < COMPLETENESS_THRESHOLD).astype(int)

df_demand_features = df_demand_features.merge(reporting_counts[["year_month","commodity","countries_reporting","demand_data_completeness","demand_data_incomplete"]], on=["year_month","commodity"], how="left")

df_demand_features.loc[df_demand_features["demand_data_incomplete"] == 1, "demand_pressure_score"] = np.nan

print(f"\nDemand features final shape: {df_demand_features.shape}")

# ── Save ───────────────────────────────────────────────────────────
save_to_lakehouse(df_demand_features, "features_demand")

# ── Verification ───────────────────────────────────────────────────
print("\n=== Incomplete months flagged ===")
incomplete = df_demand_features[df_demand_features["demand_data_incomplete"] == 1][["year_month","commodity","countries_reporting","demand_data_completeness"]]
if incomplete.empty:
    print("No incomplete months found")
else:
    print(incomplete.to_string(index=False))

print("\n=== Data completeness check last 6 months ===")
print(df_demand_features[df_demand_features["year_month"] >= "2026-01-01"][["year_month","commodity","countries_reporting","demand_data_completeness","demand_data_incomplete","demand_pressure_score"]].sort_values(["commodity","year_month"]).to_string(index=False))

print("\n=== Top 5 highest demand pressure months per commodity ===")
for commodity in COMMODITIES:
    top5 = df_demand_features[df_demand_features["commodity"] == commodity].nlargest(5,"demand_pressure_score")[["year_month","demand_pressure_score","demand_surge_ratio","countries_surging","top_buyer_country","concentrated_surge_flag","demand_data_incomplete"]]
    print(f"\n{commodity}:")
    print(top5.to_string(index=False))

print("\n=== Individually tracked countries per commodity ===")
for commodity in COMMODITIES:
    val = df_demand_features[df_demand_features["commodity"] == commodity]["individually_tracked_countries"].iloc[0]
    print(f"{commodity}: {val}")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 16, Finished, Available, Finished, False)

Comtrade rows after filtering: 35490
Commodities: ['Wheat' 'Corn' 'Rice' 'Soybean' 'Barley']
Date range: 2021-01-01 → 2026-06-01

Wheat: 129 countries, 53 months
  Individually tracked for Wheat: ['China', 'Indonesia']

Corn: 132 countries, 53 months
  Individually tracked for Corn: ['China', 'Japan', 'Rep. of Korea']

Rice: 133 countries, 53 months
  Individually tracked for Rice: ['Malaysia', 'Philippines']

Soybean: 135 countries, 65 months
  Individually tracked for Soybean: ['Brazil', 'China', 'USA']

Barley: 132 countries, 66 months
  Individually tracked for Barley: ['Australia', 'China', 'France']
Normal reporting counts (median):
commodity  normal_reporting_count
   Barley                    84.0
     Corn                    91.0
     Rice                    93.0
  Soybean                   100.0
    Wheat                    81.0

Demand features final shape: (289, 58)
✓ srm.features_demand: 289 rows saved

=== Incomplete months flagged ===
year_month commodity  countries_repo

In [15]:
# Check if recent months have lower
# country coverage than earlier months

print(df_comtrade[
    (df_comtrade["year_month"] >= "2026-01-01") &
    (df_comtrade["cmdCode"].astype(str).str.startswith(("1001","1005","1006")))
].groupby(["year_month","cmdCode"])["reporterDesc"] \
 .nunique().reset_index() \
 .rename(columns={"reporterDesc":"countries_reporting"}) \
 .to_string(index=False))

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 17, Finished, Available, Finished, False)

year_month cmdCode  countries_reporting
2026-01-01    1001                   54
2026-01-01    1005                   58
2026-01-01    1006                   61
2026-02-01    1001                   49
2026-02-01    1005                   51
2026-02-01    1006                   55
2026-03-01    1001                   47
2026-03-01    1005                   48
2026-03-01    1006                   50
2026-04-01    1001                   15
2026-04-01    1005                   12
2026-04-01    1006                   15
2026-05-01    1001                    1
2026-05-01    1005                    1
2026-05-01    1006                    1


In [16]:
save_to_lakehouse(df_demand_features,   "features_demand")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 18, Finished, Available, Finished, False)

✓ srm.features_demand: 289 rows saved


**_<u>Section 6 — Policy Risk Features</u>_**

In [17]:
# ══════════════════════════════════════════════════════════════════
# SECTION 6: POLICY RISK FEATURES
# Note: Policy augmentation and MAJOR_EXPORTERS are
#       already handled in Section 2
#       df_policy is already augmented and deduplicated
#       MAJOR_EXPORTERS is already derived from PSD data
# ═════════════════════════════════════════════════════════════════

def build_policy_features(df_policy, spine, major_exporters):
    """
    Builds policy risk features from policy ban list.
    Covers:
    - Export restriction score (indicator 5 — weight 22%)
    - Exporter restriction frequency (indicator 6 — weight 6%)

    Fixes applied:
    1. Colombia sugar quota misclassified as grain — removed
    2. Turkey oil ban misclassified as corn — removed
    3. Major exporter weighting — minor exporters at 20% weight
    4. Description-based commodity validation
    5. Oil policy detection and exclusion
    6. Missing 2022 critical events — already added in Section 2
    """

    POLICY_CUTOFF = pd.Timestamp("2026-03-01")
    SPINE_END = spine["year_month"].max()
    all_months = pd.date_range(spine["year_month"].min(), SPINE_END, freq="MS")

    SEVERITY_BASE = {"ban": 10, "quota": 7, "tax": 5, "licensing": 3, "other": 2}

    COMMODITY_KEYWORDS = {
        "Wheat": ["wheat","meslin","flour","semolina","maida","wholemeal","aata","wheat flour","wheat grain"],
        "Corn":  ["corn grain","maize grain","corn meal","maize meal","corn starch","maize starch","yellow corn","white corn","corn (maize)"],
        "Rice":  ["rice","paddy","broken rice","rice groats","parboiled rice","milled rice","husked rice","unmilled rice","brown rice","basmati"],
        "Soybean":   ["soy","soybean","soybeans","soya","soya bean","soybean meal","soymeal","oilseed"]   # NEW — deliberately excludes "soybean oil"
    }

    OIL_KEYWORDS = ["sunflower oil","soybean oil","palm oil","rapeseed oil","cottonseed oil","corn oil","vegetable oil","edible oil","crude oil","refined oil"]
    SUGAR_KEYWORDS = ["raw sugar","refined sugar","white sugar","sugar cane","sugar beet","cane sugar"]
    OTHER_EXCLUSION_KEYWORDS = ["potatoes","cotton","coffee","cocoa","tobacco","fertilizer","fuel","energy"]
    GRAIN_CONFIRM_KEYWORDS = ["wheat","corn","maize","rice","barley","grain","cereal","flour","paddy","soy","soybean","soybeans","soya"]

    # ── Step 1: Data quality validation ───────────────────────────
    def validate_policy(row):
        desc = str(row["description"]).lower() if pd.notna(row["description"]) else ""
        is_sugar_desc = any(kw in desc for kw in SUGAR_KEYWORDS)
        has_grain_desc = any(kw in desc for kw in GRAIN_CONFIRM_KEYWORDS)
        if is_sugar_desc and not has_grain_desc:
            return "misclassified_sugar"
        oil_count = sum(1 for kw in OIL_KEYWORDS if kw in desc)
        grain_count = sum(1 for kw in GRAIN_CONFIRM_KEYWORDS if kw in desc)
        if oil_count >= 2 and grain_count == 0:
            return "misclassified_oil"
        KNOWN_BAD_IDS = [109]
        if row["policy_id"] in KNOWN_BAD_IDS:
            return "misclassified_known"
        if row["issuing_country"] == "Turkey":
            if "corn oil" in desc and "corn grain" not in desc and "maize" not in desc:
                return "misclassified_oil"
        if any(kw in desc for kw in OTHER_EXCLUSION_KEYWORDS):
            if not has_grain_desc:
                return "misclassified_other"
        return "valid"

    print("=== Step 1: Data quality validation ===")
    df_policy_val = df_policy.copy()
    df_policy_val["validation_status"] = df_policy_val.apply(validate_policy, axis=1)
    print(f"Total policies: {len(df_policy_val)}")
    print(df_policy_val["validation_status"].value_counts().to_string())

    excluded = df_policy_val[df_policy_val["validation_status"] != "valid"]
    print(f"\nExcluded: {len(excluded)} policies")
    if not excluded.empty:
        print(excluded[["policy_id","issuing_country","commodity","policy_type","validation_status"]].to_string(index=False))

    df_policy_clean = df_policy_val[df_policy_val["validation_status"] == "valid"].copy()
    print(f"Clean policy rows: {len(df_policy_clean)}")

    # ── Step 2: Parse description ──────────────────────────────────
    def parse_description(row):
        desc = str(row["description"]).lower() if pd.notna(row["description"]) else ""
        result = {}
        tax_matches = re.findall(r'(\d+)\s*%', desc)
        if tax_matches and str(row["policy_type"]).lower() == "tax":
            result["tax_rate_pct"] = max([int(x) for x in tax_matches])
        else:
            result["tax_rate_pct"] = 0
        result["is_full_restriction"] = int(any(w in desc for w in ["banned","prohibited","suspend","ban","prohibition","restrict"]))
        result["is_policy_lifted"]    = int(any(w in desc for w in ["lifted","removed","ended","terminated","eased"]))
        result["is_temporary"]        = int(any(w in desc for w in ["temporary","briefly","short-term","until end of year","six-month","3-month"]))
        result["climate_driven"]      = int(any(w in desc for w in ["monsoon","drought","rainfall","flood","weather","climate","harvest","crop failure"]))
        result["shortage_driven"]     = int(any(w in desc for w in ["shortage","scarcity","supply","domestic","food security","inflation","prices","reserves"]))
        return result

    # ── Step 3: Compute severity ───────────────────────────────────
    def compute_severity(row):
        ptype = str(row["policy_type"]).strip().lower()
        base = SEVERITY_BASE.get(ptype, 2)
        if ptype == "tax" and row.get("tax_rate_pct", 0) > 0:
            if row["tax_rate_pct"] >= 20:   base = min(base + 2, 10)
            elif row["tax_rate_pct"] >= 10:  base = min(base + 1, 10)
        if row.get("is_policy_lifted", 0) == 1:
            base = base * 0.3
        if row.get("is_temporary", 0) == 1:
            base = base * 0.8
        if pd.isna(row["end_date"]):
            m_duration = 1.5
        else:
            days = (row["end_date"] - row["start_date"]).days
            if days > 365:   m_duration = 1.4
            elif days > 90:  m_duration = 1.1
            else:            m_duration = 0.8
        if pd.notna(row["announcement_date"]) and pd.notna(row["start_date"]):
            notice = (row["start_date"] - row["announcement_date"]).days
            if notice <= 1:   m_sudden = 1.5
            elif notice <= 7: m_sudden = 1.2
            else:             m_sudden = 1.0
        else:
            m_sudden = 1.0
        return min(round(base * m_duration * m_sudden, 2), 10)

    # ── Step 4: Decay rate ─────────────────────────────────────────
    def get_decay_rate(row):
        if row.get("climate_driven", 0) == 1:
            return 0.05
        elif row.get("shortage_driven", 0) == 1:
            return 0.01
        else:
            return 0.02

    # ── Step 5: Extract commodity ──────────────────────────────────
    def extract_commodities(commodity_text, description_text):
        commodity_lower = str(commodity_text).lower() if pd.notna(commodity_text) else ""
        matched = []
        for commodity, keywords in COMMODITY_KEYWORDS.items():
            commodity_match = any(kw in commodity_lower for kw in keywords)
            general_match = commodity.lower() in commodity_lower
            if commodity_match or general_match:
                matched.append(commodity)
        all_keywords = ["all commodities","all agricultural","all food","all types","all grains","all cereals"]
        if any(kw in commodity_lower for kw in all_keywords):
            matched.append("All")
        return matched if matched else ["Unknown"]

    # ── Step 6: Apply parsing and severity ────────────────────────
    print("\n=== Step 6: Applying description parsing ===")
    desc_features = df_policy_clean.apply(parse_description, axis=1, result_type="expand")
    df_policy_enriched = pd.concat([df_policy_clean.reset_index(drop=True), desc_features], axis=1)
    df_policy_enriched["severity"] = df_policy_enriched.apply(compute_severity, axis=1)
    print(f"Severity range: {df_policy_enriched['severity'].min():.2f} — {df_policy_enriched['severity'].max():.2f}")
    print(f"Temporary: {df_policy_enriched['is_temporary'].sum()} | Climate: {df_policy_enriched['climate_driven'].sum()} | Shortage: {df_policy_enriched['shortage_driven'].sum()} | Lifted: {df_policy_enriched['is_policy_lifted'].sum()}")

    # ── Step 7: Expand multi-commodity rows ───────────────────────
    print("\n=== Step 7: Expanding commodity rows ===")
    expanded_rows = []
    for _, row in df_policy_enriched.iterrows():
        commodities = extract_commodities(row["commodity"], row["description"])
        for commodity in commodities:
            new_row = row.copy()
            new_row["commodity_clean"] = commodity
            expanded_rows.append(new_row)

    df_expanded = pd.DataFrame(expanded_rows)

    all_rows = df_expanded[df_expanded["commodity_clean"] == "All"].copy()
    individual_rows = []
    for _, row in all_rows.iterrows():
        for commodity in COMMODITIES:
            new_row = row.copy()
            new_row["commodity_clean"] = commodity
            individual_rows.append(new_row)

    df_expanded = pd.concat([
        df_expanded[df_expanded["commodity_clean"] != "All"],
        pd.DataFrame(individual_rows)
    ], ignore_index=True)

    df_expanded = df_expanded[df_expanded["commodity_clean"].isin(COMMODITIES)].copy()

    print(f"Policy events after expansion: {len(df_expanded)}")
    print(df_expanded.groupby(["commodity_clean","policy_type"]).size().reset_index(name="count").to_string(index=False))

    # ── Step 8: Major exporter flag ───────────────────────────────
    print("\n=== Step 8: Adding major exporter flags ===")
    df_expanded["is_major_exporter"] = df_expanded.apply(
        lambda r: int(r["issuing_country"] in major_exporters.get(r["commodity_clean"],[])), axis=1
    )
    print(f"Major exporter policies: {df_expanded['is_major_exporter'].sum()}")
    print(f"Minor exporter policies: {(df_expanded['is_major_exporter']==0).sum()}")
    print("\nMajor exporter breakdown:")
    print(df_expanded[df_expanded["is_major_exporter"]==1].groupby(["commodity_clean","issuing_country","policy_type"]).size().reset_index(name="count").to_string(index=False))

    # ── Step 9: Map to monthly spine ──────────────────────────────
    print("\n=== Step 9: Mapping to monthly spine ===")
    monthly_records = []

    for _, event in df_expanded.iterrows():
        if pd.isna(event["start_date"]):
            continue
        is_open_ended = pd.isna(event["end_date"])
        effective_end = SPINE_END if is_open_ended else min(event["end_date"], SPINE_END)
        active_months = [m for m in all_months if event["start_date"] <= m <= effective_end]
        decay_rate = get_decay_rate(event)

        for month in active_months:
            months_active = (month - event["start_date"]).days / 30
            standard_decay = max(0.5, 1.0 - (months_active * decay_rate))
            if is_open_ended and month > POLICY_CUTOFF:
                months_beyond = (month - POLICY_CUTOFF).days / 30
                extra_decay = max(0.30, 1.0 - (months_beyond * 0.05))
            else:
                extra_decay = 1.0
            final_decay = standard_decay * extra_decay
            decayed_severity = event["severity"] * final_decay
            weighted_severity = decayed_severity if event["is_major_exporter"] == 1 else decayed_severity * 0.20

            monthly_records.append({
                "year_month":            month,
                "commodity":             event["commodity_clean"],
                "issuing_country":       event["issuing_country"],
                "policy_type":           event["policy_type"],
                "status":                event["status"],
                "severity":              event["severity"],
                "decayed_severity":      decayed_severity,
                "weighted_severity":     weighted_severity,
                "is_major_exporter":     event["is_major_exporter"],
                "tax_rate_pct":          event.get("tax_rate_pct", 0),
                "is_full_restriction":   event.get("is_full_restriction", 0),
                "climate_driven":        event.get("climate_driven", 0),
                "shortage_driven":       event.get("shortage_driven", 0),
                "is_temporary":          event.get("is_temporary", 0),
                "is_open_ended":         int(is_open_ended),
                "policy_data_confirmed": int(month <= POLICY_CUTOFF),
                "policy_id":             event["policy_id"]
            })

    df_monthly = pd.DataFrame(monthly_records)

    if df_monthly.empty:
        print("WARNING: No policy events mapped")
        return spine.copy()

    print(f"Monthly policy records: {len(df_monthly)}")

    # ── Step 10: Aggregate ─────────────────────────────────────────
    policy_agg = df_monthly.groupby(["year_month","commodity"]).agg(
        policy_risk_score           = ("decayed_severity","sum"),
        policy_risk_score_weighted  = ("weighted_severity","sum"),
        active_restrictions         = ("policy_id","nunique"),
        countries_restricting       = ("issuing_country","nunique"),
        max_single_severity         = ("decayed_severity","max"),
        major_exporter_restrictions = ("is_major_exporter","sum"),
        major_countries_restricting = ("is_major_exporter", lambda x: (x==1).sum()),
        active_bans    = ("policy_type", lambda x: (x.str.lower()=="ban").sum()),
        active_quotas  = ("policy_type", lambda x: (x.str.lower()=="quota").sum()),
        active_taxes   = ("policy_type", lambda x: (x.str.lower()=="tax").sum()),
        full_restrictions      = ("is_full_restriction","sum"),
        climate_driven_events  = ("climate_driven","sum"),
        shortage_driven_events = ("shortage_driven","sum"),
        policy_data_confirmed  = ("policy_data_confirmed","min")
    ).reset_index()

    policy_agg["policy_risk_score"]          = policy_agg["policy_risk_score"].clip(0, 100)
    policy_agg["policy_risk_score_weighted"] = policy_agg["policy_risk_score_weighted"].clip(0, 100)
    policy_agg["multi_country_shock"]        = (policy_agg["major_countries_restricting"] >= 2).astype(int)
    policy_agg["broad_country_shock"]        = (policy_agg["countries_restricting"] >= 5).astype(int)

    # ── Step 11: Escalation trend ──────────────────────────────────
    policy_agg = policy_agg.sort_values(["commodity","year_month"])
    policy_agg["policy_3m_trend"]   = policy_agg.groupby("commodity")["policy_risk_score_weighted"].diff(3)
    policy_agg["policy_escalating"] = (policy_agg["policy_3m_trend"] > 10).astype(int)

    # ── Step 12: Rolling 12m restriction events ───────────────────
    restriction_events = df_monthly.groupby(["year_month","commodity"])["policy_id"].nunique().reset_index()
    restriction_events.columns = ["year_month","commodity","monthly_restriction_events"]
    restriction_events = restriction_events.sort_values(["commodity","year_month"])
    restriction_events["restriction_events_12m"] = restriction_events.groupby("commodity")["monthly_restriction_events"].transform(lambda x: x.rolling(12).sum())
    policy_agg = policy_agg.merge(restriction_events[["year_month","commodity","restriction_events_12m"]], on=["year_month","commodity"], how="left")

    # ── Step 13: Fill months with no active policies ───────────────
    full_spine = spine.copy()
    policy_agg = full_spine.merge(policy_agg, on=["year_month","commodity"], how="left")

    fill_cols = [
        "policy_risk_score","policy_risk_score_weighted",
        "active_restrictions","countries_restricting",
        "max_single_severity","major_exporter_restrictions",
        "major_countries_restricting","active_bans",
        "active_quotas","active_taxes","full_restrictions",
        "climate_driven_events","shortage_driven_events",
        "multi_country_shock","broad_country_shock",
        "policy_escalating","restriction_events_12m"
    ]
    for col in fill_cols:
        if col in policy_agg.columns:
            policy_agg[col] = policy_agg[col].fillna(0)

    policy_agg["policy_data_confirmed"] = policy_agg["policy_data_confirmed"].fillna(1)

    policy_cols = [
        "year_month","commodity",
        "policy_risk_score","policy_risk_score_weighted",
        "active_restrictions","countries_restricting",
        "major_exporter_restrictions","major_countries_restricting",
        "max_single_severity","active_bans","active_quotas","active_taxes",
        "full_restrictions","climate_driven_events","shortage_driven_events",
        "multi_country_shock","broad_country_shock",
        "policy_escalating","restriction_events_12m",
        "policy_data_confirmed","policy_3m_trend"
    ]
    policy_cols = [c for c in policy_cols if c in policy_agg.columns]
    return policy_agg[policy_cols]


# ── Run — df_policy already augmented from Section 2 ──────────────
# df_policy_features = build_policy_features(df_policy, spine_train, MAJOR_EXPORTERS)
df_policy_features = build_policy_features(df_policy, spine_full, MAJOR_EXPORTERS)

save_to_lakehouse(df_policy_features, "features_policy")
print(f"\nPolicy features shape: {df_policy_features.shape}")

# ── Sanity checks ──────────────────────────────────────────────────
print("\n=== Sanity Check 1: Russia wheat ban (Mar-Aug 2022) ===")
print(df_policy_features[
    (df_policy_features["commodity"]=="Wheat") &
    (df_policy_features["year_month"]>="2022-03-01") &
    (df_policy_features["year_month"]<="2022-08-01")
][[
    "year_month","policy_risk_score","policy_risk_score_weighted",
    "major_exporter_restrictions","major_countries_restricting",
    "active_bans","multi_country_shock"
]].to_string(index=False))

print("\n=== Sanity Check 2: India rice ban (Jul-Oct 2023) ===")
print(df_policy_features[
    (df_policy_features["commodity"]=="Rice") &
    (df_policy_features["year_month"]>="2023-07-01") &
    (df_policy_features["year_month"]<="2023-10-01")
][[
    "year_month","policy_risk_score","policy_risk_score_weighted",
    "major_exporter_restrictions","major_countries_restricting",
    "multi_country_shock"
]].to_string(index=False))

print("\n=== Top 5 weighted policy risk months — Wheat ===")
print(df_policy_features[df_policy_features["commodity"]=="Wheat"].nlargest(5,"policy_risk_score_weighted")[[
    "year_month","policy_risk_score","policy_risk_score_weighted",
    "major_exporter_restrictions","major_countries_restricting","multi_country_shock"
]].to_string(index=False))

print("\n=== Top 5 weighted policy risk months — Rice ===")
print(df_policy_features[df_policy_features["commodity"]=="Rice"].nlargest(5,"policy_risk_score_weighted")[[
    "year_month","policy_risk_score","policy_risk_score_weighted",
    "major_exporter_restrictions","major_countries_restricting","multi_country_shock"
]].to_string(index=False))

print("\n=== Policy risk distribution ===")
print(df_policy_features.groupby("commodity")[["policy_risk_score","policy_risk_score_weighted"]].describe().round(2))

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 19, Finished, Available, Finished, False)

=== Step 1: Data quality validation ===
Total policies: 120
validation_status
valid                  110
misclassified_other      7
misclassified_sugar      2
misclassified_oil        1

Excluded: 10 policies
 policy_id issuing_country                                                                                                commodity policy_type   validation_status
        22         Morocco                                                                               Tomatoes, onions, potatoes         Ban misclassified_other
        52      Uzbekistan                                                          Cotton seed oil, sunflower oil, sunflower seeds         Ban misclassified_other
        63         Belarus                                                                                                    sugar   Licensing misclassified_sugar
        85          Turkey sunflower oil, soybean oil, sunflower seeds, cottonseed oils, rapeseed, mustard oil, corn oil, margarine    

In [18]:
# # Check exact country names in policy table
# # for Russia and Ukraine
# print("Russia/Ukraine entries in policy table:")
# print(df_policy[
#     df_policy["issuing_country"].str.contains(
#         "Russia|Ukraine|Russian", case=False, na=False
#     )
# ][["policy_id","issuing_country","commodity","policy_type","start_date"]].to_string(index=False))

# # Check exact country names in PSD major exporters
# print("\nRussia/Ukraine in MAJOR_EXPORTERS:")
# for commodity, countries in MAJOR_EXPORTERS.items():
#     russia = [c for c in countries if "russia" in c.lower() or "ukraine" in c.lower()]
#     if russia:
#         print(f"  {commodity}: {russia}")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 20, Finished, Available, Finished, False)

In [19]:
# # Check India wheat policies
# print("India wheat policies in table:")
# print(df_policy[
#     (df_policy["issuing_country"] == "India") &
#     (df_policy["commodity"].str.lower().str.contains("wheat", na=False))
# ][["policy_id","issuing_country","commodity","policy_type","start_date","end_date"]].to_string(index=False))

# # Check what 2022 wheat policies exist at all
# print("\nAll wheat policies starting in 2022:")
# print(df_policy[
#     (df_policy["start_date"] >= "2022-01-01") &
#     (df_policy["start_date"] <= "2022-12-31") &
#     (df_policy["commodity"].str.lower().str.contains("wheat", na=False))
# ][["policy_id","issuing_country","commodity","policy_type","start_date","end_date"]].to_string(index=False))

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 21, Finished, Available, Finished, False)

In [20]:
# # Investigate Corn 2025 policy events
# print("Active corn policies Jul 2025:")
# print(df_policy[
#     (df_policy["commodity"].str.lower().str.contains("corn|maize", na=False)) &
#     (df_policy["start_date"] <= pd.Timestamp("2025-07-01")) &
#     ((df_policy["end_date"] >= pd.Timestamp("2025-07-01")) | df_policy["end_date"].isna())
# ][["issuing_country","policy_type","start_date","end_date","description"]].to_string(index=False))

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 22, Finished, Available, Finished, False)

In [21]:
dc_barley_aligned = df_wasde[
    (df_wasde["commodity"]=="Barley") &
    (df_wasde["year_month"] >= "2021-01-01") &
    (df_wasde["year_month"] <= "2025-12-01")
]["stocks_to_use_ratio"]
print(dc_barley_aligned.describe())

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 23, Finished, Available, Finished, False)

count    59.000000
mean     12.137627
std       0.616700
min      11.100000
25%      11.770000
50%      12.080000
75%      12.495000
max      13.860000
Name: stocks_to_use_ratio, dtype: float64


**_<u>Section 7 — KSA concentration features</u>_**

In [22]:
# ══════════════════════════════════════════════════════════════════
# SECTION 7: KSA CONCENTRATION FEATURES
# Covers: KSA top 3 dependency (indicator 7 — weight 8%)
# Source: gld_ksa_import_export (ANNUAL data)
# Uses: salic_classification column for commodity mapping
# Excludes: Seeds (corn seed) and Other Animal Feed (mixed feed)
# Fix 1: trade_type handles both Import and Imports
# Fix 2: Country name normalisation
# Fix 3: ksa_concentration_rising forward filled for future months
# ══════════════════════════════════════════════════════════════════

def build_ksa_features(df_ksa, spine):
    """
    Builds KSA import concentration features.
    Annual data expanded to monthly.
    Uses lag1 — what you KNEW entering the year.
    """

    # NEW
    SALIC_COMMODITY_MAP = {"Wheat": "Wheat", "Corn": "Corn", "Rice": "Rice", "Soybeans": "Soybean", "Barley": "Barley"}

    COUNTRY_NAME_FIX = {
        "RUSSIAN FEDERATION": "Russia",
        "RUSSIA":             "Russia",
        "U.S.A":              "USA",
        "U.S.A.":             "USA",
        "US":                 "USA",
        "UNITED STATES":      "USA",
        "INDIA":              "India",
        "PAKISTAN":           "Pakistan",
        "ARGENTINA":          "Argentina",
        "BRAZIL":             "Brazil",
        "UKRAINE":            "Ukraine",
        "GERMANY":            "Germany",
        "CANADA":             "Canada",
        "POLAND":             "Poland",
        "AUSTRALIA":          "Australia",
        "LITHUANIA":          "Lithuania",
        "LATVIA":             "Latvia",
        "ROMANIA":            "Romania",
        "THAILAND":           "Thailand",
        "VIETNAM":            "Vietnam",
        "VIETNAM ":           "Vietnam",
        "SERBIA":             "Serbia",
        "PARAGUAY":           "Paraguay"
    }

    # ── Fix 1: Handle both Import and Imports ──────────────────────
    df_ksa_imp = df_ksa[df_ksa["trade_type"].isin(["Import","Imports"])].copy()
    print(f"trade_type values found: {df_ksa_imp['trade_type'].unique()}")

    # ── Apply salic_classification mapping ─────────────────────────
    df_ksa_imp["commodity_model"] = df_ksa_imp["salic_classification"].map(SALIC_COMMODITY_MAP)
    df_ksa_filtered = df_ksa_imp[df_ksa_imp["commodity_model"].isin(COMMODITIES)].copy()
    df_ksa_filtered["weight_mt"] = df_ksa_filtered["weight"] / 1000

    # ── Fix 2: Normalise country names ────────────────────────────
    df_ksa_filtered["country"] = df_ksa_filtered["country"].str.strip().replace(COUNTRY_NAME_FIX)

    print(f"\nKSA filtered rows: {len(df_ksa_filtered)}")
    print(f"Years available: {sorted(df_ksa_filtered['year'].unique())}")
    print(f"\nCommodity volume breakdown (MT):")
    print(df_ksa_filtered.groupby("commodity_model")["weight_mt"].sum().reset_index().to_string(index=False))

    print(f"\nTop 10 KSA Wheat import sources (all years):")
    print(df_ksa_filtered[df_ksa_filtered["commodity_model"]=="Wheat"].groupby("country")["weight_mt"].sum().nlargest(10).reset_index().to_string(index=False))

    print(f"\nTop 10 KSA Rice import sources (all years):")
    print(df_ksa_filtered[df_ksa_filtered["commodity_model"]=="Rice"].groupby("country")["weight_mt"].sum().nlargest(10).reset_index().to_string(index=False))

    print(f"\nTop 10 KSA Corn import sources (all years):")
    print(df_ksa_filtered[df_ksa_filtered["commodity_model"]=="Corn"].groupby("country")["weight_mt"].sum().nlargest(10).reset_index().to_string(index=False))

    # ── Compute annual features ────────────────────────────────────
    annual_features = []

    for commodity in COMMODITIES:
        for year in sorted(df_ksa_filtered["year"].unique()):
            dc = df_ksa_filtered[
                (df_ksa_filtered["commodity_model"] == commodity) &
                (df_ksa_filtered["year"] == year)
            ]

            if dc.empty:
                continue

            country_vol = dc.groupby("country")["weight_mt"].sum()
            total_vol   = country_vol.sum()

            if total_vol == 0:
                continue

            shares = country_vol / total_vol

            hhi            = (shares ** 2).sum()
            top3           = shares.nlargest(3).sum()
            top1_country   = shares.idxmax()
            top1_share     = shares.max()
            n_sources      = (shares > 0.01).sum()
            top3_countries = shares.nlargest(3).index.tolist()

            annual_features.append({
                "year":                 year,
                "commodity":            commodity,
                "ksa_hhi":              round(hhi, 4),
                "ksa_top3_share":       round(top3, 4),
                "ksa_top1_country":     top1_country,
                "ksa_top1_share":       round(top1_share, 4),
                "ksa_top3_countries":   str(top3_countries),
                "ksa_n_sources":        int(n_sources),
                "ksa_total_imports_mt": round(total_vol, 2)
            })

    df_annual = pd.DataFrame(annual_features)
    df_annual = df_annual.sort_values(["commodity","year"]).reset_index(drop=True)

    print(f"\nAnnual features shape: {df_annual.shape}")
    print(f"Years per commodity:")
    print(df_annual.groupby("commodity")["year"].count())

    # ── Lag1 features ──────────────────────────────────────────────
    df_annual["ksa_hhi_lag1"]             = df_annual.groupby("commodity")["ksa_hhi"].shift(1)
    df_annual["ksa_top3_share_lag1"]      = df_annual.groupby("commodity")["ksa_top3_share"].shift(1)
    df_annual["ksa_top1_share_lag1"]      = df_annual.groupby("commodity")["ksa_top1_share"].shift(1)
    df_annual["ksa_top1_country_lag1"]    = df_annual.groupby("commodity")["ksa_top1_country"].shift(1)
    df_annual["ksa_n_sources_lag1"]       = df_annual.groupby("commodity")["ksa_n_sources"].shift(1)

    # YoY change features
    df_annual["ksa_hhi_yoy"]              = df_annual.groupby("commodity")["ksa_hhi"].diff()
    df_annual["ksa_top3_share_yoy"]       = df_annual.groupby("commodity")["ksa_top3_share"].diff()
    df_annual["ksa_concentration_rising"] = (df_annual["ksa_hhi_yoy"] > 0.02).astype(int)
    df_annual["ksa_diversifying"]         = (df_annual.groupby("commodity")["ksa_n_sources"].diff() > 0).astype(int)

    # ── Expand annual to monthly ───────────────────────────────────
    monthly_spine = spine.copy()
    monthly_spine["year"] = monthly_spine["year_month"].dt.year

    df_ksa_monthly = monthly_spine.merge(df_annual, on=["year","commodity"], how="left")
    df_ksa_monthly = df_ksa_monthly.sort_values(["commodity","year_month"])

    # ── Fix 3: Forward fill including concentration flags ──────────
    ksa_fill_cols = [
        "ksa_hhi","ksa_top3_share","ksa_top1_share","ksa_n_sources",
        "ksa_hhi_lag1","ksa_top3_share_lag1",
        "ksa_top1_share_lag1","ksa_n_sources_lag1",
        "ksa_concentration_rising","ksa_diversifying",  # ← added
        "ksa_hhi_yoy","ksa_top3_share_yoy"              # ← added
    ]
    df_ksa_monthly[ksa_fill_cols] = df_ksa_monthly.groupby("commodity")[ksa_fill_cols].ffill()

    # Fill any remaining NULLs with 0 for flag columns
    flag_cols = ["ksa_concentration_rising","ksa_diversifying"]
    df_ksa_monthly[flag_cols] = df_ksa_monthly[flag_cols].fillna(0)

    ksa_cols = [
        "year_month","commodity",
        "ksa_hhi","ksa_top3_share",
        "ksa_top1_country","ksa_top1_share",
        "ksa_top3_countries","ksa_n_sources",
        "ksa_total_imports_mt",
        "ksa_hhi_lag1","ksa_top3_share_lag1",
        "ksa_top1_share_lag1","ksa_top1_country_lag1",
        "ksa_n_sources_lag1",
        "ksa_hhi_yoy","ksa_top3_share_yoy",
        "ksa_concentration_rising","ksa_diversifying"
    ]
    ksa_cols = [c for c in ksa_cols if c in df_ksa_monthly.columns]
    return df_ksa_monthly[ksa_cols], df_annual


# ── Run ────────────────────────────────────────────────────────────
# df_ksa_features, df_ksa_annual = build_ksa_features(df_ksa, spine_train)
df_ksa_features, df_ksa_annual = build_ksa_features(df_ksa, spine_full)

save_to_lakehouse(df_ksa_features, "features_ksa")
save_to_lakehouse(df_ksa_annual,   "ksa_features_annual")

print(f"\nKSA features shape:  {df_ksa_features.shape}")
print(f"KSA annual shape:    {df_ksa_annual.shape}")

# ── Checks ────────────────────────────────────────────────────────
print("\n=== KSA top 3 sources per commodity (most recent year) ===")
latest_year = df_ksa_annual["year"].max()
print(df_ksa_annual[df_ksa_annual["year"]==latest_year][[
    "commodity","year","ksa_hhi","ksa_top3_share",
    "ksa_top3_countries","ksa_n_sources"
]].to_string(index=False))

print("\n=== KSA concentration trend over years ===")
for commodity in COMMODITIES:
    print(f"\n{commodity}:")
    print(df_ksa_annual[df_ksa_annual["commodity"]==commodity][[
        "year","ksa_hhi","ksa_top3_share",
        "ksa_n_sources","ksa_top1_country"
    ]].to_string(index=False))

print("\n=== Monthly feature sample (lag1 values) ===")
print(df_ksa_features[[
    "year_month","commodity",
    "ksa_hhi_lag1","ksa_top3_share_lag1",
    "ksa_n_sources_lag1","ksa_concentration_rising"
]].tail(9).to_string(index=False))

print("\n=== NULL check on all key features ===")
check_cols = [
    "ksa_hhi_lag1","ksa_top3_share_lag1",
    "ksa_n_sources_lag1","ksa_concentration_rising",
    "ksa_diversifying","ksa_hhi_yoy"
]
print(df_ksa_features[check_cols].isnull().sum())

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 24, Finished, Available, Finished, False)

trade_type values found: ['Import' 'Imports']

KSA filtered rows: 2756
Years available: [2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]

Commodity volume breakdown (MT):
commodity_model    weight_mt
         Barley 97346228.197
           Corn 49681107.375
           Rice 21145613.789
        Soybean  9021141.616
          Wheat 41613455.928

Top 10 KSA Wheat import sources (all years):
  country   weight_mt
   Russia 8129382.729
  Germany 6954293.990
   Poland 5985878.695
Lithuania 4473526.730
  Romania 3740771.417
   Canada 2446828.527
   Brazil 1796797.832
   Latvia 1703355.632
Australia 1468364.070
  Ukraine 1343875.790

Top 10 KSA Rice import sources (all years):
   country    weight_mt
     India 15646635.913
  Pakistan  2032144.344
       USA  1635163.436
  Thailand   835107.962
 Australia   366801.195
   Vietnam   305630.375
    Brazil   102891.387
     EGYPT    39081.022
  CAMBODIA    26682.124
BANGLADESH    25843.270

Top 10 KSA Corn

**_<u>Section 8 — Master Merge</u>_**

In [23]:
# ══════════════════════════════════════════════════════════════════
# SECTION 8: MASTER FEATURE TABLE
# Merges all 5 feature tables onto training spine
# One row per year_month × commodity
# Output: srm.features_master
# ══════════════════════════════════════════════════════════════════

def build_master_features(spine, df_wasde_features, df_bdi_features,
                           df_demand_features, df_policy_features,
                           df_ksa_features):
    """
    Joins all feature sets onto master spine.
    One row per year_month × commodity.

    Join strategy:
    WASDE   → has commodity dimension → join on year_month + commodity
    BDI     → no commodity dimension → join on year_month only (same for all)
    Demand  → has commodity dimension → join on year_month + commodity
    Policy  → has commodity dimension → join on year_month + commodity
    KSA     → has commodity dimension → join on year_month + commodity
    """

    print("=== Section 8: Master Feature Merge ===")
    print(f"Spine shape: {spine.shape}")

    df = spine.copy()

    # ── Join 1: WASDE features ─────────────────────────────────────
    wasde_cols = [c for c in df_wasde_features.columns
                  if c not in ["year_month","commodity"]]
    df = df.merge(
        df_wasde_features[["year_month","commodity"] + wasde_cols],
        on=["year_month","commodity"],
        how="left"
    )
    print(f"After WASDE join:   {df.shape}")

    # ── Join 2: BDI features ───────────────────────────────────────
    # BDI has no commodity dimension
    # Same BDI values apply to all commodities in the same month
    bdi_cols = [c for c in df_bdi_features.columns if c != "year_month"]
    df = df.merge(
        df_bdi_features[["year_month"] + bdi_cols],
        on="year_month",
        how="left"
    )
    print(f"After BDI join:     {df.shape}")

    # ── Join 3: Demand features ────────────────────────────────────
    demand_cols = [c for c in df_demand_features.columns
                   if c not in ["year_month","commodity"]]
    df = df.merge(
        df_demand_features[["year_month","commodity"] + demand_cols],
        on=["year_month","commodity"],
        how="left"
    )
    print(f"After Demand join:  {df.shape}")

    # ── Join 4: Policy features ────────────────────────────────────
    policy_cols = [c for c in df_policy_features.columns
                   if c not in ["year_month","commodity"]]
    df = df.merge(
        df_policy_features[["year_month","commodity"] + policy_cols],
        on=["year_month","commodity"],
        how="left"
    )
    print(f"After Policy join:  {df.shape}")

    # ── Join 5: KSA features ───────────────────────────────────────
    ksa_cols = [c for c in df_ksa_features.columns
                if c not in ["year_month","commodity"]]
    df = df.merge(
        df_ksa_features[["year_month","commodity"] + ksa_cols],
        on=["year_month","commodity"],
        how="left"
    )
    print(f"After KSA join:     {df.shape}")

    # ── Fill policy NULLs with zero ────────────────────────────────
    # Months with no active policies = zero risk
    policy_fill_cols = [
        "policy_risk_score","policy_risk_score_weighted",
        "active_restrictions","countries_restricting",
        "major_exporter_restrictions","major_countries_restricting",
        "max_single_severity","active_bans","active_quotas",
        "active_taxes","full_restrictions","climate_driven_events",
        "shortage_driven_events","multi_country_shock",
        "broad_country_shock","policy_escalating",
        "restriction_events_12m"
    ]
    for col in policy_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # ── Fill demand NULLs with zero ────────────────────────────────
    # Months with no surge = zero pressure
    demand_fill_cols = [
        "demand_surge_ratio","demand_mom_change",
        "countries_surging","top_buyer_surging",
        "top_buyer_surge_ratio","concentrated_surge_flag",
        "demand_pressure_score"
    ]
    for col in demand_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # ── Sort ───────────────────────────────────────────────────────
    df = df.sort_values(["commodity","year_month"]).reset_index(drop=True)

    return df


# ── Run ────────────────────────────────────────────────────────────
# Training features
df_master_train = build_master_features(
    spine_train,
    df_wasde_features, df_bdi_features,
    df_demand_features, df_policy_features,
    df_ksa_features
)

# Drop high NULL cols
cols_to_drop = ["stu_3y_avg","stu_vs_3y_avg"]
df_master_train = df_master_train.drop(
    columns=[c for c in cols_to_drop if c in df_master_train.columns]
)

save_to_lakehouse(df_master_train, "features_master_train")
print(f"Training features: {df_master_train.shape}")

# Scoring features (Jan-Jun 2026)
df_master_score = build_master_features(
    spine_score,
    df_wasde_features, df_bdi_features,
    df_demand_features, df_policy_features,
    df_ksa_features
)

df_master_score = df_master_score.drop(
    columns=[c for c in cols_to_drop if c in df_master_score.columns]
)

save_to_lakehouse(df_master_score, "features_master_score")
print(f"Scoring features:  {df_master_score.shape}")

# Keep df_master as full table for reference
df_master = pd.concat(
    [df_master_train, df_master_score],
    ignore_index=True
).sort_values(["commodity","year_month"])

save_to_lakehouse(df_master, "features_master")
print(f"Full master:       {df_master.shape}")

print(f"Date range: {df_master['year_month'].min().date()} → {df_master['year_month'].max().date()}")
print(f"Rows per commodity:\n{df_master.groupby('commodity').size()}")

# ── Column inventory ───────────────────────────────────────────────
print(f"\nTotal features: {len(df_master.columns) - 2}")
print("\nAll columns:")
for i, col in enumerate(df_master.columns):
    print(f"  {i+1:3d}. {col}")

# ── NULL analysis ──────────────────────────────────────────────────
print("\n=== NULL count per column ===")
null_counts = df_master.isnull().sum()
null_pct    = (null_counts / len(df_master) * 100).round(1)
null_summary = pd.DataFrame({
    "null_count": null_counts,
    "null_pct":   null_pct
})
print(null_summary[null_summary["null_count"] > 0].sort_values(
    "null_pct", ascending=False
).to_string())

# ── Data completeness by commodity ────────────────────────────────
print("\n=== Key feature completeness by commodity ===")
key_features = [
    "stu_ratio","ppi_base","ppi_score",
    "bdi","bdi_z_score",
    "demand_pressure_score",
    "policy_risk_score","policy_risk_score_weighted",
    "ksa_hhi_lag1","ksa_top3_share_lag1"
]
for commodity in COMMODITIES:
    dc = df_master[df_master["commodity"]==commodity]
    nulls = dc[key_features].isnull().sum()
    print(f"\n{commodity} ({len(dc)} rows):")
    print(nulls.to_string())

# ── Sanity checks on master table ─────────────────────────────────
print("\n=== Sanity Check: Ukraine War Wheat (Mar-Jun 2022) ===")
print(df_master[
    (df_master["commodity"]=="Wheat") &
    (df_master["year_month"]>="2022-03-01") &
    (df_master["year_month"]<="2022-06-01")
][[
    "year_month","stu_ratio","ppi_score",
    "policy_risk_score_weighted","major_exporter_restrictions",
    "bdi_z_score","demand_pressure_score"
]].to_string(index=False))

print("\n=== Sanity Check: India Rice Ban (Jul-Oct 2023) ===")
print(df_master[
    (df_master["commodity"]=="Rice") &
    (df_master["year_month"]>="2023-07-01") &
    (df_master["year_month"]<="2023-10-01")
][[
    "year_month","stu_ratio","ppi_score",
    "policy_risk_score_weighted","major_exporter_restrictions",
    "ksa_hhi_lag1","demand_pressure_score"
]].to_string(index=False))

print("\n=== Feature summary statistics ===")
summary_cols = [
    "stu_ratio","ppi_score","bdi",
    "demand_pressure_score",
    "policy_risk_score_weighted",
    "ksa_hhi_lag1","ksa_top3_share_lag1"
]
print(df_master.groupby("commodity")[summary_cols].mean().round(2).to_string())

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 25, Finished, Available, Finished, False)

=== Section 8: Master Feature Merge ===
Spine shape: (360, 2)
After WASDE join:   (360, 21)
After BDI join:     (360, 38)
After Demand join:  (360, 94)
After Policy join:  (360, 113)
After KSA join:     (360, 129)
✓ srm.features_master_train: 360 rows saved
Training features: (360, 129)
=== Section 8: Master Feature Merge ===
Spine shape: (30, 2)
After WASDE join:   (30, 21)
After BDI join:     (30, 38)
After Demand join:  (30, 94)
After Policy join:  (30, 113)
After KSA join:     (30, 129)
✓ srm.features_master_score: 30 rows saved
Scoring features:  (30, 129)
✓ srm.features_master: 390 rows saved
Full master:       (390, 129)
Date range: 2020-01-01 → 2026-06-01
Rows per commodity:
commodity
Barley     78
Corn       78
Rice       78
Soybean    78
Wheat      78
dtype: int64

Total features: 127

All columns:
    1. year_month
    2. commodity
    3. stu_ratio
    4. stu_mom_change
    5. stu_3m_avg
    6. stu_6m_avg
    7. stu_yoy_change
    8. stu_stress_flag
    9. stu_sustained_stre

In [24]:
# Remove policy_3m_trend from MODEL_FEATURE_COLS
# It has 37% NULLs — too many for reliable model feature
# policy_escalating already captures the same signal

MODEL_FEATURE_COLS = [
    # STU features
    "stu_ratio","stu_mom_change","stu_3m_avg",
    "stu_6m_avg","stu_yoy_change",
    "stu_stress_flag","stu_sustained_stress",
    # PPI features
    "ppi_base","ppi_score","ppi_3m_trend",
    "prod_mom_revision","prod_rev_pct",
    "prod_cut_flag","consecutive_prod_cuts",
    # Stock features
    "stocks_mom_change","stocks_3m_trend",
    "stocks_drawdown_flag","months_of_cover",
    "export_to_prod_ratio",
    # BDI features
    "bdi","bdi_mom_pct","bdi_3m_avg",
    "bdi_6m_avg","bdi_12m_avg",
    "bdi_yoy_pct","bdi_z_score",
    "bdi_vs_3m_avg","bdi_vs_6m_avg",
    "bdi_spike_flag","bdi_sustained_high",
    "bdi_3m_direction","bdi_3m_volatility",
    # Demand features
    "demand_surge_ratio","demand_mom_change",
    "countries_surging","top_buyer_surge_ratio",
    "top_buyer_surging","concentrated_surge_flag",
    "demand_pressure_score",
    # Policy features — removed policy_3m_trend
    "policy_risk_score","policy_risk_score_weighted",
    "active_restrictions","countries_restricting",
    "major_exporter_restrictions","major_countries_restricting",
    "max_single_severity","active_bans",
    "active_quotas","active_taxes",
    "full_restrictions","climate_driven_events",
    "shortage_driven_events","multi_country_shock",
    "broad_country_shock","policy_escalating",
    "restriction_events_12m",
    # KSA features
    "ksa_hhi_lag1","ksa_top3_share_lag1",
    "ksa_top1_share_lag1","ksa_n_sources_lag1",
    "ksa_hhi_yoy","ksa_top3_share_yoy",
    "ksa_concentration_rising","ksa_diversifying"
]

print(f"Updated MODEL_FEATURE_COLS: {len(MODEL_FEATURE_COLS)} features")

# Recheck NULLs with updated feature list
print("\n=== TRAINING NULL check — updated feature list ===")
model_nulls = df_master_train[MODEL_FEATURE_COLS].isnull().sum()
has_nulls = model_nulls[model_nulls > 0]
if len(has_nulls) > 0:
    print(has_nulls.to_string())
else:
    print("All model features clean ✓")

# Resave feature list
import json
import os
# os.makedirs("Files/config", exist_ok=True)
# with open("Files/config/feature_cols.json","w") as f:
#     json.dump(MODEL_FEATURE_COLS, f, indent=2)

content = json.dumps(MODEL_FEATURE_COLS, indent=2)

# Ensure folder exists in Lakehouse Files
notebookutils.fs.mkdirs("Files/config")

# Write JSON into Lakehouse Files
notebookutils.fs.put(
    "Files/config/feature_cols.json",
    content,
    overwrite=True
)

print(f"\n✓ Updated feature list saved: {len(MODEL_FEATURE_COLS)} features")

# Show what NULLs remain and how many rows will survive label creation
print("\n=== Rows that will survive after dropping NULL key features ===")
key_training_cols = ["stu_ratio","ppi_base","ppi_score","bdi"]
for commodity in COMMODITIES:
    dc = df_master_train[df_master_train["commodity"]==commodity]
    clean = dc.dropna(subset=key_training_cols)
    print(f"{commodity}: {len(dc)} total → {len(clean)} clean rows after dropping NULLs")

print("\n=== NOTEBOOK 2 TRULY COMPLETE ===")
print("Proceed to Notebook 3 — Label Creation")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 26, Finished, Available, Finished, False)

Updated MODEL_FEATURE_COLS: 64 features

=== TRAINING NULL check — updated feature list ===
stu_ratio                 48
stu_mom_change            53
stu_3m_avg                58
stu_6m_avg                73
stu_yoy_change           108
stu_stress_flag           48
stu_sustained_stress      58
ppi_base                 108
ppi_score                108
ppi_3m_trend             123
prod_mom_revision         53
prod_rev_pct              53
prod_cut_flag             48
consecutive_prod_cuts     58
stocks_mom_change         53
stocks_3m_trend           63
stocks_drawdown_flag      48
months_of_cover           48
export_to_prod_ratio     120

✓ Updated feature list saved: 64 features

=== Rows that will survive after dropping NULL key features ===
Wheat: 72 total → 48 clean rows after dropping NULLs
Corn: 72 total → 48 clean rows after dropping NULLs
Rice: 72 total → 48 clean rows after dropping NULLs
Soybean: 72 total → 48 clean rows after dropping NULLs
Barley: 72 total → 60 clean rows afte

**New Section 9 - FOB Price features:**

In [25]:
def build_price_features(df_price_intl, df_price_te):
    """
    Builds FOB Price raw features (indicator 8 — weight 10%)
    Covers: Wheat, Corn, Barley, Soybean (FOB origin assessments)
             Rice (TE global benchmark, RR1:COM)

    Rising FOB price = costlier imports = supply stress
    Falling FOB price = cheaper imports = easier supply flow
    """
    PRICE_SERIES_MAP = {
        "Wheat":   {"source": "intl", "name": "Wheat 12.5% FOB Russia USD/mt"},
        "Corn":    {"source": "intl", "name": "Corn FOB Argentina USD/mt"},
        "Barley":  {"source": "intl", "name": "Barley Feed barley FOB Russia USD/mt"},
        "Soybean": {"source": "intl", "name": "Soybean FOB Brazil Santos USD/mt"},
        "Rice":    {"source": "te",   "code": "RR1:COM"},
    }

    all_rows = []
    for commodity, spec in PRICE_SERIES_MAP.items():
        if spec["source"] == "intl":
            dc = df_price_intl[df_price_intl["Name"] == spec["name"]].copy()
            dc["ds"] = pd.to_datetime(dc["Date"])
        else:
            dc = df_price_te[df_price_te["Code"] == spec["code"]].copy()
            dc["ds"] = pd.to_datetime(dc["Date"], format="%m/%d/%Y")

        dc = dc[["ds", "Price"]].rename(columns={"Price": "price"}).dropna(subset=["price"])
        dc = dc.sort_values("ds")
        dc["year_month"] = dc["ds"].dt.to_period("M").dt.to_timestamp()

        monthly = dc.groupby("year_month").agg(price=("price", "last")).reset_index()
        monthly["commodity"] = commodity
        all_rows.append(monthly)

    df_price_monthly = pd.concat(all_rows, ignore_index=True)
    return df_price_monthly.sort_values(["commodity", "year_month"]).reset_index(drop=True)

df_price_features = build_price_features(df_price_intl, df_price_te)
save_to_lakehouse(df_price_features, "features_price")

StatementMeta(, 7cdd014c-e28d-4c3d-99ea-e747ba7b5b78, 27, Finished, Available, Finished, False)

✓ srm.features_price: 524 rows saved
